# Musanga V4 — NB01: Combined exhaustive plot + KVO circle reference and leak-proof spatial folds

**New V4 field65 workflow:** The inherited Sections 01–16 preserve the legacy V3_1 circle/partial reference and RGB/DSM/MS diagnostics. **Sections 17–20 are the new authoritative V4 combined-data reference and train/validation manifest.** The original Section 11 field stem-count registry and old fold/window contracts are for historical comparison, not the V4 reference. NB02 must explicitly consume the new V4 field65 contract and manifest; old NB02 models do not automatically switch training targets.

**Section 18A** audits repeated plot/circle Musanga labels (including GIS disagreements) and reassigns indivisible overlapping annotation groups to the same spatial fold before Section 19 creates masks.


## 01 — Drive environment and audited input registry


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib
from pathlib import Path
import json, math, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from rasterio.windows import Window, from_bounds
from rasterio.features import rasterize
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
DATA=next((p for p in [DRIVE/'Colab Notebooks/Musanga/Final/Data',DRIVE/'Musanga/Final/Data'] if p.exists()),None)
assert DATA is not None,'Final/Data directory not found'
AUDIT=DATA.parent/'Outputs/V3_1/00_input_audit'  # approved reference, read-only
V4_AUDIT=DATA.parent/'Outputs/v4/00_input_audit/00_v4_input_audit_manifest.json'
assert V4_AUDIT.exists(), 'Run V4 NB00 first'
V4_INFO=json.loads(V4_AUDIT.read_text())
assert V4_INFO.get('schema_version')=='v4_nb00_rgb_dsm_sfm_inventory_1',V4_INFO.get('schema_version')
OUT=DATA.parent/'Outputs/v4/01_reference_data_no_MOS34';OUT.mkdir(parents=True,exist_ok=True)
manifest_path=AUDIT/'00_input_manifest.json'
assert manifest_path.exists(),'Run Notebook 00 through Block 10 first'
A=json.loads(manifest_path.read_text())
EXCLUDED_MOSAIC='Mos_3_4_transparent_mosaic_group1_georef.tif'
EXCLUDED_IDS={6,9,24,25}
assert EXCLUDED_MOSAIC in [r['file'] for r in A['mosaics']]
MOSAICS={r['file']:r for r in A['mosaics'] if r['file']!=EXCLUDED_MOSAIC}
assert A.get('schema_version')=='v3_1_input_audit_kvo_final', (
    'NB00 Nadine reference manifest required; rerun NB00 Block 10')
TREE_FILES={site:Path(p) for site,p in A['shared_annotations'].items()}
ROI_FILE=Path(A['roi'])
CIRCLE_FILE=Path(A['exhaustive_annotations']['complete_roi_boundaries'])
EXH_FILE=Path(A['exhaustive_annotations']['positive_polygons'])
SITES=['Baego','MOS','Yangambi','Yoko']
EXPECTED_CIRCLES=int(A['exhaustive_annotations']['verified_exhaustive_circle_count'])
EXPECTED_ROWS=set(map(int,A['exhaustive_annotations']['verified_exhaustive_circle_rows']))
assert EXPECTED_CIRCLES==34 and len(EXPECTED_ROWS)==34, (
    'This revision is pinned to the newly audited 34 Nadine circles')
assert CIRCLE_FILE.name.casefold()=='circles_roi_nadine.shp'
assert EXH_FILE.name.casefold()=='circles_annotations_kvo_final.shp'
assert A['exhaustive_annotations'].get('all_annotation_polygons_musanga') is True
assert len(MOSAICS)==5 and len(TREE_FILES)==4
assert set(TREE_FILES)==set(SITES)
print('V4 inherits V3_1 field-grounded KVO final; all exhaustive polygons treated as Musanga positives')
print('30 retained circles; same V3 spatial folds; MOS 3-4 excluded.')
assert A.get('reference_version')=='KVO_field_final'
V3_FOLDS=DATA.parent/'Outputs/V3/01_reference_data_no_MOS34/01_circle_spatial_folds.csv'
assert V3_FOLDS.exists(), 'Original V3 fold mapping required for matched comparison.'
V3_SPLIT=pd.read_csv(V3_FOLDS)
approved_repairs={(r['version'],int(r['source_row'])):float(r['relative_area_change'])
                  for r in A['annotation_versioning'].get('provisionally_accepted_geometry_repairs',[])}
FOLDS=3;PATCH=512;BUFFER_M=15.;SEED=20261015
for p in [ROI_FILE,CIRCLE_FILE,EXH_FILE,*TREE_FILES.values(),*[Path(r['path']) for r in MOSAICS.values()]]:
    assert p.exists(),p
print('Found',len(MOSAICS),'RGB mosaics and',len(TREE_FILES),'shared tree annotation files')
print('Read-only input:',DATA,'\nWrites:',OUT)


## 02 — Repair geometry individually, with traceable quarantines

Invalid geometries repaired in memory if relative area change <= 5% for partial tree polygons and <= 1% for ROI/exhaustive reference polygons; otherwise excluded individually, never all polygons from the same file. Review quarantines before relying on the final outputs.


In [ ]:
REPAIRS=[]
def parts(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in parts(q)]
    return []
def load_polygons(path):
    max_change=.05 if Path(path) in TREE_FILES.values() else .01
    raw=gpd.read_file(path)
    assert raw.crs is not None,path
    rows=[];geoms=[]
    for idx,g in enumerate(raw.geometry):
        status='OK';delta=0.
        if g is None or g.is_empty:status='EMPTY'
        elif not g.is_valid or g.geom_type not in ('Polygon','MultiPolygon'):
            try:
                fixed=unary_union(parts(make_valid(g)))
                delta=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if not fixed.is_valid or fixed.is_empty:status='QUARANTINE'
                elif delta>max_change:
                    key=('v2',idx)
                    allowed=(Path(path)==EXH_FILE and key in approved_repairs
                             and abs(delta-approved_repairs[key])<0.0005)
                    if not allowed:status='QUARANTINE'
                    else:
                        g=fixed;status='REPAIRED_APPROVED'
                else:g=fixed;status='REPAIRED'
            except Exception:status='QUARANTINE'
        REPAIRS.append(dict(file=path.name,row=idx,status=status,area_change=delta))
        if status in ('OK','REPAIRED','REPAIRED_APPROVED'):rows.append(idx);geoms.append(g)
    output=raw.iloc[rows].copy().reset_index(drop=True)
    output.geometry=geoms
    output['_row_original']=rows
    print(path.name,':',len(output),'kept,',len(raw)-len(output),'quarantined')
    return output
ROI=load_polygons(ROI_FILE);CIRCLES=load_polygons(CIRCLE_FILE)
EXH=load_polygons(EXH_FILE)
TREES={site:load_polygons(path) for site,path in TREE_FILES.items()}
pd.DataFrame(REPAIRS).to_csv(OUT/'01_geometry_repairs.csv',index=False)
assert ROI.geometry.is_valid.all()
for site,g in TREES.items():
    assert 'Class_ID' in g.columns,site
    cls=pd.to_numeric(g.Class_ID,errors='coerce')
    g['_label']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1)).astype('int8')
    print(site,':',int((g._label==1).sum()),'Musanga /',int((g._label==0).sum()),'known negatives')
# The 34 circles were already certified exhaustive in V3.
# A v2 circle with zero Musanga polygons is a valid observed zero,
# not a reason to drop that evaluation circle.
exh_union=unary_union(EXH.to_crs(CIRCLES.crs).geometry.to_list())
verified=[i for i,row in enumerate(CIRCLES['_row_original'])
          if int(row) in EXPECTED_ROWS]
assert set(CIRCLES['_row_original'].iloc[verified].astype(int))==EXPECTED_ROWS
CIRCLES=CIRCLES.iloc[verified].copy().reset_index(drop=True)
CIRCLES['circle_id']=CIRCLES['_row_original'].astype(int)
assert len(CIRCLES)==EXPECTED_CIRCLES,(
    f'Expected {EXPECTED_CIRCLES} verified Nadine circles; got {len(CIRCLES)}')
assert len(EXH)==int(A['exhaustive_annotations']['musanga_polygons']), (
    'Polygon count changed or repairs were quarantined; inspect before proceeding')
assert set(CIRCLES.circle_id)==EXPECTED_ROWS
print('All',len(EXH),'exhaustive polygons retained, regardless of Class_ID')
print('Verified raw exhaustive circles:',list(CIRCLES.circle_id))
assert EXCLUDED_IDS.issubset(set(CIRCLES.circle_id))
CIRCLES=CIRCLES[~CIRCLES.circle_id.isin(EXCLUDED_IDS)].copy()
assert len(CIRCLES)==30
print('30 retained circles; excluded:',sorted(EXCLUDED_IDS))


## 03 — Six mosaics nested within four forest groups; hard general ROI


In [ ]:
REG=[];GEO={}
for name,info in MOSAICS.items():
    with rasterio.open(info['path']) as ds:
        assert ds.crs and ds.crs.is_projected and ds.count>=3,name
        footprint=box(*ds.bounds)
        roi=ROI.to_crs(ds.crs)
        eligible=unary_union(roi.geometry.to_list()).intersection(footprint)
        REG.append(dict(mosaic=name,site=info['site'],roi_overlap_m2=float(eligible.area),
                        width=ds.width,height=ds.height,pixel_m=abs(ds.res[0]),crs=str(ds.crs)))
        GEO[name]=dict(transform=ds.transform,crs=ds.crs,width=ds.width,height=ds.height,
                       footprint=footprint,roi=roi,site=info['site'])
REG=pd.DataFrame(REG);display(REG)
REG.to_csv(OUT/'01_mosaic_registry.csv',index=False)
assert (REG.roi_overlap_m2>0).all(),'At least one mosaic lacks ROI coverage: resolve ROI.shp before continuing'


## 03A — V4 RGB–DSM pairing and grid metadata (new; V3_1 blocks unchanged)

This section pairs the existing SfM DSMs with their RGB mosaics using metadata only. **No DSM data outside ROI.shp is sampled or evaluated.** The ROI-aware valid-pixel checks take place in Block 04A after the original V3_1 circle-to-mosaic assignment and frozen spatial-fold logic.


In [ ]:
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.windows import Window
RGB_TO_DSM_NAME={
  'Baego.tif':'Baego_DSM.tif',
  'Mos_1_2_transparent_mosaic_group1_georef.tif':'Mos_1_2_dsm_georef.tif',
  'Mos_5_6_transparent_mosaic_group1_georef.tif':'mlt_Mos_5_6_dsm_georef (2).tif',
  'Yangambi_georef.tif':'Yangambi_georef_DSM.tif',
  'Yoko_kvo_georef.tif':'Yoko_kvo_georef_DSM.tif',
}
# Reuse the exact paths inventoried by V4 NB00 rather than guessing DSM directories.
inv_path=DATA.parent/'Outputs/v4/00_input_audit/00_file_inventory.csv'
assert inv_path.is_file(), 'NB00 inventory missing'
file_inv=pd.read_csv(inv_path)
assert set(MOSAICS)==set(RGB_TO_DSM_NAME), 'V3_1 five-RGB contract unexpectedly changed'
DSM_PATHS={}
dsm_rows=[]
for rgb_name,dsm_filename in RGB_TO_DSM_NAME.items():
    found=file_inv[file_inv['name'].astype(str).str.casefold()==dsm_filename.casefold()]
    assert len(found)==1,(dsm_filename,found[['path']].to_dict('records'))
    dsm_path=Path(found.iloc[0]['path'])
    assert dsm_path.exists(),dsm_path
    DSM_PATHS[rgb_name]=str(dsm_path)
    with rasterio.open(MOSAICS[rgb_name]['path']) as rgb, rasterio.open(dsm_path) as dsm:
        assert rgb.crs and dsm.crs and rgb.crs==dsm.crs, (rgb_name,rgb.crs,dsm.crs)
        assert dsm.count==1,(dsm_filename,dsm.count)
        from rasterio.warp import transform_bounds
        db=transform_bounds(dsm.crs,rgb.crs,*dsm.bounds,densify_pts=21)
        left=max(rgb.bounds.left,db[0]);bottom=max(rgb.bounds.bottom,db[1])
        right=min(rgb.bounds.right,db[2]);top=min(rgb.bounds.top,db[3])
        area=max(0,right-left)*max(0,top-bottom)
        rgb_area=(rgb.bounds.right-rgb.bounds.left)*(rgb.bounds.top-rgb.bounds.bottom)
        frac=area/max(rgb_area,1e-10)
        samegrid=(rgb.width==dsm.width and rgb.height==dsm.height
                  and np.allclose(tuple(rgb.transform),tuple(dsm.transform),atol=1e-8,rtol=0))
        dsm_rows.append(dict(rgb=rgb_name,dsm=dsm_filename,dsm_path=str(dsm_path),
            rgb_resolution_m=float(abs(rgb.transform.a)),dsm_resolution_m=float(abs(dsm.transform.a)),
            same_grid=bool(samegrid),bbox_intersection_fraction=float(frac),crs=str(dsm.crs)))
DSM_META=pd.DataFrame(dsm_rows)
DSM_META.to_csv(OUT/'01_rgb_dsm_pairing_metadata.csv',index=False)
display(DSM_META)
print('Pixel-grid matching is not required (MOS differs). DSM reads will be windowed and aligned to RGB.')
# Keep a stable DSM path contract for downstream NB02B and NB03.
(OUT/'01_dsm_file_registry.json').write_text(json.dumps({
    'schema':'v4_dsm_registry_1','rgb_to_dsm':DSM_PATHS,
    'alignment':'WarpedVRT windowed on exact RGB affine/grid; bilinear; avoid full mosaic reprojection',
    'elevation_interpretation':'SfM visible canopy elevation, NOT CHM',
    'registration_status':'metadata only; crown-level image inspection needed'
},indent=2))

def dsm_window_on_rgb_grid(rgb_path,dsm_path,window):
    # Each call reads ONLY the geographic window needed, reprojected to native RGB grid.
    with rasterio.open(rgb_path) as rgb, rasterio.open(dsm_path) as dsm:
        with WarpedVRT(dsm,crs=rgb.crs,transform=rgb.transform,
                       width=rgb.width,height=rgb.height,
                       resampling=Resampling.bilinear) as aligned:
            return aligned.read(1,window=window,masked=True)

# Valid-data diagnostics are intentionally deferred to Block 04A, where the
# exact V3_1 ROI + retained reference circles + valid RGB mask are available.
print('DSM file/grid registry saved. ROI-only DSM coverage audit follows Block 04.')


## 04 — Circle-to-mosaic ownership and spatial CV

First audit all 34 Nadine circles against **both** each orthomosaic footprint and general ROI.shp. NB00 tested image bounding boxes; a circle can be fully inside imagery but outside the modelling ROI. Save a full diagnostic and stop if any circle is under 95% covered. Only after resolving discrepancies, build three spatial folds per forest group.


In [ ]:
# Audit footprint coverage separately from general ROI coverage, for EVERY circle.
# NB00 checked orthomosaic bounding-box overlap; that does NOT prove ROI coverage.
roi_unions={name:unary_union(d['roi'].geometry.to_list()) for name,d in GEO.items()}
LINK_ROWS=[]
for _,row in CIRCLES.iterrows():
    candidates=[]
    for name,d in GEO.items():
        g=gpd.GeoSeries([row.geometry],crs=CIRCLES.crs).to_crs(d['crs']).iloc[0]
        bbox_area=float(g.intersection(d['footprint']).area)
        in_roi_area=float(g.intersection(d['footprint']).intersection(roi_unions[name]).area)
        candidates.append((bbox_area,in_roi_area,name))
    # Assign the mosaic by image-footprint overlap, NOT ROI overlap.
    # This identifies circles that have imagery but lie outside ROI.shp.
    bbox_area,roi_area,name=max(candidates,key=lambda v:(v[0],v[1]))
    circle_area=float(row.geometry.area)
    LINK_ROWS.append(dict(circle_id=int(row.circle_id),site=GEO[name]['site'],mosaic=name,
        circle_area_m2=circle_area,imagery_bbox_area_m2=bbox_area,
        imagery_bbox_coverage_fraction=bbox_area/circle_area,
        roi_intersection_area_m2=roi_area,roi_coverage_fraction=roi_area/circle_area,
        cover_pct=100*row.geometry.intersection(exh_union).area/circle_area))
COVERAGE=pd.DataFrame(LINK_ROWS)
COVERAGE.to_csv(OUT/'01_all_circle_roi_coverage_diagnostic.csv',index=False)
print('=== RETAINED 30-CIRCLE ROI AND MOSAIC COVERAGE DIAGNOSTIC ===')
display(COVERAGE.round(4))
problem=COVERAGE[(COVERAGE.imagery_bbox_coverage_fraction<.95) |
                 (COVERAGE.roi_coverage_fraction<.95)].copy()
print('=== CIRCLES REQUIRING ATTENTION (<95% coverage) ===')
display(problem.round(4))
print('Diagnostic saved:',OUT/'01_all_circle_roi_coverage_diagnostic.csv')
if len(problem):
    print('These circles must not be silently dropped or assigned negative outside ROI.shp.')
    raise AssertionError(
        'Reference circles have insufficient image and/or general ROI coverage: '+
        str(problem.circle_id.tolist())+
        '. Inspect ROI.shp against Nadine circle boundaries. '
        'NB00 bounding-box overlap alone was insufficient.')
LINK=COVERAGE[['circle_id','site','mosaic','roi_coverage_fraction','cover_pct']].copy()
expected_site_counts={'Baego':7,'MOS':11,'Yangambi':7,'Yoko':5}
assert LINK.groupby('site').size().to_dict()==expected_site_counts,(
    'New circle ownership unexpectedly differs from audited 34-circle dataset')
assert LINK.circle_id.is_unique and len(LINK)==30
# Keep the original V3 fold for every circle: annotation quality is the ONLY
# experimental variable. Do not recompute folds from polygons or centroids.
assert V3_SPLIT.circle_id.is_unique
original_folds=V3_SPLIT[['circle_id','site','fold']].copy()
assert set(original_folds.circle_id)==set(LINK.circle_id)
LINK=LINK.merge(original_folds[['circle_id','fold']],on='circle_id',
                validate='one_to_one')
assert (LINK[['circle_id','site']].merge(original_folds,on='circle_id',
        suffixes=('_new','_v3')).eval('site_new == site_v3')).all(), (
        'Circle-to-site mapping changed from V3')
assert LINK.groupby('site').fold.nunique().eq(FOLDS).all()
assert LINK.groupby(['site','fold']).circle_id.nunique().ge(1).all()
assert LINK.circle_id.nunique()==30
print('Per-site / fold circle counts:')
display(LINK.groupby(['site','fold']).circle_id.nunique().unstack(fill_value=0))
display(LINK.sort_values(['site','fold','circle_id']).round(3))
LINK.to_csv(OUT/'01_circle_spatial_folds.csv',index=False)


## 04A — DSM coverage exclusively on the V3_1 reference evaluation support

For each retained circle, evaluate **circle ∩ ROI.shp ∩ valid RGB pixels**. The denominator is *only* this original evaluation support, never the full circle, raster bounding box, or areas outside ROI. Missing DSM and near-zero elevation diagnostics are calculated only inside the support. No changes to circle membership, labels, or folds.


In [ ]:
# V4-only diagnostics, on the SAME spatial support as V3_1 evaluations:
# retained circle ∩ general ROI.shp ∩ valid RGB image pixels.
# No whole-mosaic reads; each circle uses a local 128x128 sampling grid.
from rasterio.transform import from_bounds as affine_from_bounds
from rasterio.features import rasterize
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

CIRCLE_GRID=128
coverage_rows=[]
for name,dsm_path in DSM_PATHS.items():
    associated=LINK.loc[LINK.mosaic==name,['circle_id','site','fold']]
    with rasterio.open(dsm_path) as dsm, rasterio.open(MOSAICS[name]['path']) as rgb:
        assert dsm.crs == rgb.crs, (name,dsm.crs,rgb.crs)
        # ROI in this mosaic's CRS, exactly the ROI objects loaded by V3_1 NB01.
        roi_geom=roi_unions[name]
        for _,ref in associated.iterrows():
            circle=CIRCLES.loc[CIRCLES.circle_id==int(ref.circle_id)].to_crs(rgb.crs).geometry.iloc[0]
            support_geom=circle.intersection(roi_geom).intersection(box(*rgb.bounds))
            if support_geom.is_empty or support_geom.area<=0:
                raise AssertionError(f'Circle {ref.circle_id} has no ROI/RGB footprint support')
            tr=affine_from_bounds(*circle.bounds,CIRCLE_GRID,CIRCLE_GRID)
            roi_circle_mask=rasterize([(support_geom,1)],
                 out_shape=(CIRCLE_GRID,CIRCLE_GRID),transform=tr,
                 fill=0,dtype='uint8').astype(bool)
            # Sample the RGB validity mask on the same local diagnostic grid.
            # dataset_mask respects alpha/NoData, unlike checking just band 1.
            with WarpedVRT(rgb,crs=rgb.crs,transform=tr,
                           width=CIRCLE_GRID,height=CIRCLE_GRID,
                           resampling=Resampling.nearest) as rgb_local:
                rgb_valid=rgb_local.dataset_mask()>0
            support=roi_circle_mask & rgb_valid
            denom=int(support.sum())
            if denom==0:
                raise AssertionError(f'Circle {ref.circle_id} has zero valid RGB within ROI')
            with WarpedVRT(dsm,crs=rgb.crs,transform=tr,
                           width=CIRCLE_GRID,height=CIRCLE_GRID,
                           resampling=Resampling.bilinear) as dsm_local:
                z=dsm_local.read(1,masked=True)
            numeric=np.asarray(z.data,dtype=np.float64)
            dsm_valid=(~np.ma.getmaskarray(z)) & np.isfinite(numeric) & support
            # Flag zeros for review; never silently rewrite source DSM.
            zeros=dsm_valid & np.isclose(numeric,0.,atol=1e-6)
            usable=dsm_valid & ~zeros
            coverage_rows.append(dict(site=ref.site,mosaic=name,circle_id=int(ref.circle_id),
                fold=int(ref.fold),dsm=Path(dsm_path).name,
                roi_rgb_support_pixels=denom,
                roi_circle_sample_pixels=int(roi_circle_mask.sum()),
                valid_rgb_fraction_in_roi_circle=denom/max(int(roi_circle_mask.sum()),1),
                dsm_valid_fraction_in_roi_rgb=float(dsm_valid.sum()/denom),
                dsm_zero_fraction_in_roi_rgb=float(zeros.sum()/denom),
                dsm_nonzero_fraction_in_roi_rgb=float(usable.sum()/denom),
                median_nonzero_z_m=float(np.median(numeric[usable])) if usable.any() else np.nan,
                dsm_nodata=dsm.nodata))
CIRCLE_DSM=pd.DataFrame(coverage_rows).sort_values(['site','circle_id'])
assert len(CIRCLE_DSM)==30 and CIRCLE_DSM.circle_id.is_unique, 'Expected all 30 original circles'
CIRCLE_DSM.to_csv(OUT/'01_dsm_circle_coverage_diagnostic.csv',index=False)
display(CIRCLE_DSM.round(3))
print('=== DSM COVERAGE ON ROI ∩ VALID RGB ∩ REFERENCE CIRCLE ONLY ===')
display(CIRCLE_DSM.groupby('site').agg(
    circles=('circle_id','count'),
    min_dsm_coverage=('dsm_nonzero_fraction_in_roi_rgb','min'),
    median_dsm_coverage=('dsm_nonzero_fraction_in_roi_rgb','median'),
    max_zero_fraction=('dsm_zero_fraction_in_roi_rgb','max')).round(3))
bad=CIRCLE_DSM[CIRCLE_DSM.dsm_nonzero_fraction_in_roi_rgb<0.95]
if len(bad):
    print('DSM <95% usable sampled coverage WITHIN ROI/RGB support:')
    display(bad[['site','circle_id','dsm_nonzero_fraction_in_roi_rgb']].round(3))
    print('Do not drop reference circles or alter V3_1 masks/folds; review DSM validity.')
else:
    print('All circles >=95% sampled DSM coverage on ROI/valid-RGB support.')
print('Circle diagnostics saved:',OUT/'01_dsm_circle_coverage_diagnostic.csv')
print('No DSM or RGB pixels outside ROI were included in these metrics.')


## 05 — Single-source raster label generator (native resolution)

`label_window()` returns target, loss weight, annotation source, uncertainty and general ROI masks for a **small native raster window**. `Class_ID=1` means Musanga; other numeric classes mean known non-Musanga. Unknown background outside circles has zero weight. Explicit contradictions have zero training weight. For evaluation, only exhaustive held-out circle pixels count, and contradictions are retained in the full-reference analysis.


In [ ]:
CACHE={}
for name,d in GEO.items():
    circles=CIRCLES.to_crs(d['crs']).copy()
    circles['assigned_mosaic']=circles.circle_id.map(LINK.set_index('circle_id').mosaic)
    circles['fold']=circles.circle_id.map(LINK.set_index('circle_id').fold)
    d['circles']=circles[circles.assigned_mosaic==name].copy()
    # Spatial protection covers all circles of this forest group,
    # even where MOS orthomosaics overlap.
    d['site_circles']=circles[circles.circle_id.isin(
        LINK.loc[LINK.site==d['site'],'circle_id'])].copy()
    d['site_circles']['fold']=d['site_circles'].circle_id.map(
        LINK.set_index('circle_id').fold)
    d['positive_exhaustive']=EXH.to_crs(d['crs'])
    d['trees']=TREES[d['site']].to_crs(d['crs'])
    CACHE[name]=d
def label_window(name,win,fold=None,role='production'):
    """role='train', 'val', 'production'. Valid pixels only inside ROI.shp."""
    d=CACHE[name]
    shape=(int(win.height),int(win.width))
    transform=rasterio.windows.transform(win,d['transform'])
    bounds=box(*rasterio.windows.bounds(win,d['transform']))
    def burn(geoms):
        intersecting=[g.intersection(bounds) for g in geoms if g is not None and g.intersects(bounds)]
        if not intersecting:return np.zeros(shape,dtype=bool)
        return rasterize([(g,1) for g in intersecting if not g.is_empty],
            out_shape=shape,transform=transform,fill=0,dtype='uint8').astype(bool)
    roi=burn(d['roi'].geometry)
    circles=d['circles']
    if role in ('train','val'):
        assert fold in range(FOLDS)
        circles=circles[circles.fold.ne(fold) if role=='train' else circles.fold.eq(fold)]
    elif role!='production':raise ValueError(role)
    exhaustive=burn(circles.geometry)&roi
    pos_ex=burn(d['positive_exhaustive'].geometry)&exhaustive
    trees=d['trees'];pos_tree=burn(trees.loc[trees._label==1].geometry)
    neg_tree=burn(trees.loc[trees._label==0].geometry)
    uncertain=(pos_ex&neg_tree)|(exhaustive&~pos_ex&pos_tree)|(~exhaustive&pos_tree&neg_tree)
    source=np.zeros(shape,dtype='uint8')
    source[(pos_tree|neg_tree)&~exhaustive]=1
    source[exhaustive]=2
    target=(pos_ex | (~exhaustive&pos_tree)).astype('uint8')
    if role=='val':
        weight=exhaustive.astype('uint8') # full reference includes disagreement for independent QC
    else:
        weight=(roi&(source>0)&~uncertain).astype('uint8')
        if role=='train':
            # also suppress labels within buffer of held-out circles
            holdout=d['site_circles'][d['site_circles'].fold==fold]
            forbidden=burn([g.buffer(BUFFER_M) for g in holdout.geometry])
            weight[forbidden]=0
    weight[~roi]=0;source[~roi]=0;target[~roi]=0
    return dict(target=target,weight=weight,source=source,
                uncertain=(uncertain&roi).astype('uint8'),roi=roi.astype('uint8'))
print('Prepared reference rasterization function for',len(CACHE),'mosaics')


## 06 — Compact patch-window candidates

Systematic windows cover exhaustive circles; partial crown windows provide supplementary training examples away from exhaustive circles. **No full-resolution RGB mosaics are copied or rasterized.** Each window points to the original TIFF. MOS crowns are assigned to one mosaic when image footprints overlap.


In [ ]:
PATCH_ROWS=[]
MAX_PARTIAL_PER_MOSAIC=300
for name,d in CACHE.items():
    W,H=d['width'],d['height']
    assert W>=PATCH and H>=PATCH
    for _,circle in d['circles'].iterrows():
        w=from_bounds(*circle.geometry.bounds,transform=d['transform'])
        r0=max(0,int(w.row_off));c0=max(0,int(w.col_off))
        r1=min(H,int(math.ceil(w.row_off+w.height)));c1=min(W,int(math.ceil(w.col_off+w.width)))
        for rr in range((r0//PATCH)*PATCH,r1,PATCH):
            for cc in range((c0//PATCH)*PATCH,c1,PATCH):
                win=Window(cc,rr,min(PATCH,W-cc),min(PATCH,H-rr))
                if not circle.geometry.intersects(box(*rasterio.windows.bounds(win,d['transform']))):continue
                PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='exhaustive',
                    circle_id=int(circle.circle_id),fold=int(circle.fold),
                    row=int(rr),col=int(cc),height=int(win.height),width=int(win.width)))
    # Partial-label patches cannot overlap circles or their exclusion buffers.
    exclusion=unary_union([g.buffer(BUFFER_M) for g in d['site_circles'].geometry]) if len(d['site_circles']) else None
    trees=d['trees'][d['trees']._label.isin([0,1])]
    rng=np.random.default_rng(SEED+sum(map(ord,name)))
    seen=set()
    for i in rng.permutation(len(trees)):
        p=trees.geometry.iloc[i].representative_point()
        if d['site']=='MOS':
            owners=sorted(k for k,v in CACHE.items() if v['site']=='MOS' and v['footprint'].covers(p))
            if owners and name!=owners[0]:continue
        rr,cc=rasterio.transform.rowcol(d['transform'],p.x,p.y)
        rr=int(max(0,min(H-PATCH,int(rr)-PATCH//2)))
        cc=int(max(0,min(W-PATCH,int(cc)-PATCH//2)))
        key=(rr//(PATCH//2),cc//(PATCH//2))
        if key in seen:continue
        win=Window(cc,rr,PATCH,PATCH)
        shape=box(*rasterio.windows.bounds(win,d['transform']))
        if exclusion is not None and shape.intersects(exclusion):continue
        if not any(g.intersects(shape) for g in d['roi'].geometry):continue
        seen.add(key)
        PATCH_ROWS.append(dict(site=d['site'],mosaic=name,kind='partial',circle_id=-1,fold=-1,
                               row=rr,col=cc,height=PATCH,width=PATCH))
        if len(seen)>=MAX_PARTIAL_PER_MOSAIC:break
PATCHES=pd.DataFrame(PATCH_ROWS).drop_duplicates()
assert len(PATCHES)>0
display(PATCHES.groupby(['site','kind'],as_index=False).size())
PATCHES.to_csv(OUT/'01_candidate_windows.csv',index=False)
print('Reference windows:',len(PATCHES),'(RGB remains in its original TIFFs)')


## 07 — Fold-specific leak-proof train/validation manifests

Entire RGB training windows crossing a held-out circle plus 15 m buffer are excluded, even if the labels outside the circle were usable. The test region stays independent of partial annotation examples. Held-out circle windows are evaluation windows only; their reference pixels are confined to the specific held-out circle.


In [ ]:
ROWS=[]
for fold in range(FOLDS):
    for _,p in PATCHES.iterrows():
        d=CACHE[p.mosaic]
        win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
        footprint=box(*rasterio.windows.bounds(win,d['transform']))
        holdout=d['site_circles'][d['site_circles'].fold==fold]
        near_holdout=any(footprint.intersects(g.buffer(BUFFER_M)) for g in holdout.geometry)
        if p.kind=='exhaustive' and int(p.fold)==fold:split='val'
        elif near_holdout:split='excluded_spatial_guard'
        else:split='train'
        ROWS.append(dict(cv_fold=fold,site=p.site,mosaic=p.mosaic,kind=p.kind,
                         circle_id=int(p.circle_id),row=int(p.row),col=int(p.col),
                         width=int(p.width),height=int(p.height),split=split))
SPLITS=pd.DataFrame(ROWS)
# Every site has held-out exhaustive reference in each fold.
counts=SPLITS[SPLITS.split=='val'].groupby(['cv_fold','site']).circle_id.nunique()
assert len(counts)==FOLDS*len(SITES) and counts.min()>=1,counts
for _,row in SPLITS[SPLITS.split=='train'].iterrows():
    d=CACHE[row.mosaic]
    geom=box(*rasterio.windows.bounds(Window(row.col,row.row,row.width,row.height),d['transform']))
    assert not any(geom.intersects(g.buffer(BUFFER_M)) for g in
                   d['site_circles'].loc[d['site_circles'].fold==row.cv_fold].geometry),'Training RGB spatial leakage'
display(SPLITS.groupby(['cv_fold','site','split'],as_index=False).size())
SPLITS.to_csv(OUT/'01_cv_window_manifest.csv',index=False)


## 08 — Exhaustive annotation QA and central-crop coverage

Select high-coverage tiles (not arbitrary row order) for visualization. Audit whether the central 256 px crop used by NB02D actually intersects each reference circle; report circles not represented in any central crop.


In [ ]:
# Block 08 — Diagnostic QC: select a genuinely representative 512 px circle window.
# NB02D trains on the CENTRAL 256 px crop of each 512 px candidate.
# Hence measure both full-window and central-crop annotation support.
from matplotlib.colors import ListedColormap
QC_ROWS=[]
circle_lookup=CIRCLES.set_index('circle_id')
for p in PATCHES.loc[PATCHES.kind=='exhaustive'].itertuples(index=False):
    geom=circle_lookup.loc[int(p.circle_id)].geometry
    d=CACHE[p.mosaic]
    # Circle and raster share CRS only after transforming to this mosaic.
    circle_local=gpd.GeoSeries([geom],crs=CIRCLES.crs).to_crs(d['crs']).iloc[0]
    win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
    full_box=box(*rasterio.windows.bounds(win,d['transform']))
    crop_size=min(256,int(win.height),int(win.width))
    center_win=Window(int(win.col_off)+(int(win.width)-crop_size)//2,
                      int(win.row_off)+(int(win.height)-crop_size)//2,
                      crop_size,crop_size)
    central_box=box(*rasterio.windows.bounds(center_win,d['transform']))
    full_area=float(circle_local.intersection(full_box).area)
    central_area=float(circle_local.intersection(central_box).area)
    QC_ROWS.append(dict(site=p.site,mosaic=p.mosaic,circle_id=int(p.circle_id),
        row=int(p.row),col=int(p.col),height=int(p.height),width=int(p.width),
        circle_area_full_m2=full_area,circle_area_center256_m2=central_area))
QC_WINDOWS=pd.DataFrame(QC_ROWS)
QC_WINDOWS['usable_center256']=QC_WINDOWS.circle_area_center256_m2.gt(0)
coverage_summary=QC_WINDOWS.groupby('site').agg(
    candidate_windows=('circle_id','size'),
    windows_with_center256_circle=('usable_center256','sum'),
    distinct_circles=('circle_id','nunique'))
coverage_summary['circles_with_center256_circle']=(
    QC_WINDOWS[QC_WINDOWS.usable_center256].groupby('site').circle_id.nunique()
).reindex(coverage_summary.index,fill_value=0).astype(int)
display(coverage_summary)
missing_circles=(set(CIRCLES.circle_id.astype(int))-
                 set(QC_WINDOWS.loc[QC_WINDOWS.usable_center256,'circle_id'].astype(int)))
print('Circles without ANY candidate window containing reference in central 256px:',sorted(missing_circles))
QC_WINDOWS.to_csv(OUT/'01_exhaustive_window_center256_QC.csv',index=False)
# Choose high-coverage tiles for the actual visual QC, rather than arbitrary median rows.
examples=[]
for site in SITES:
    group=QC_WINDOWS[(QC_WINDOWS.site==site)&(QC_WINDOWS.usable_center256)]
    assert len(group)>0,f'No representative exhaustive window for site {site}'
    examples.append(group.sort_values(
        ['circle_area_center256_m2','circle_area_full_m2'],ascending=False).iloc[0])
fig,axs=plt.subplots(len(examples),3,figsize=(12,3.3*len(examples)),constrained_layout=True)
if len(examples)==1:axs=axs[None,:]
for i,p in enumerate(examples):
    win=Window(int(p.col),int(p.row),int(p.width),int(p.height))
    lab=label_window(p.mosaic,win,role='production')
    with rasterio.open(MOSAICS[p.mosaic]['path']) as ds:
        rgb=np.moveaxis(ds.read([1,2,3],window=win,masked=True).filled(0),0,-1)
    if rgb.dtype!=np.uint8:
        scale=max(float(np.nanpercentile(rgb,99)),1.)
        rgb=(255*np.clip(rgb.astype('float32')/scale,0,1)).astype('uint8')
    mask=lab['weight']>0
    n_trusted=int(mask.sum())
    n_positive=int(((lab['target']>0)&mask).sum())
    assert n_trusted>250,f'No labelled pixels in selected QC tile: {p.site}, circle {p.circle_id}'
    print(f'{p.site}: circle {int(p.circle_id)}; trusted {n_trusted:,} pixels; '
          f'Musanga {n_positive:,} ({100*n_positive/n_trusted:.1f}%)')
    category=np.zeros_like(lab['target'])
    category[mask]=1
    category[mask&(lab['target']==1)]=2
    category[lab['uncertain']>0]=3
    category[lab['roi']==0]=4
    axs[i,0].imshow(rgb);axs[i,0].set_title(f'{p.site}: RGB (circle {int(p.circle_id)})')
    axs[i,1].imshow(category,vmin=0,vmax=4,cmap=ListedColormap(
        ['#333333','#cccccc','#2b8c4b','#efb63e','#000000']))
    axs[i,1].set_title('Unknown / non-Musanga / Musanga / conflict / outside ROI')
    axs[i,2].imshow(lab['weight'],cmap='gray',vmin=0,vmax=1)
    axs[i,2].set_title(f'Trusted: {100*mask.mean():.1f}% of tile')
    for ax in axs[i]:ax.axis('off')
fig.savefig(OUT/'01_label_QC.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)
if missing_circles:
    warnings.warn('Some circles have no exhaustive-window central-256 coverage. '
                  'Investigate candidate-window geometry before running NB02D.')


## 09 — Site and fold balance audit

The exhaustive circles are the reference for fraction accuracy. Their observed cover is reported separately by forest and CV fold. The 34 Nadine circles are not assumed to be a representative probability sample of each entire forest.


In [ ]:
STAT=LINK[['site','mosaic','circle_id','fold','roi_coverage_fraction','cover_pct']].copy()
display(STAT.groupby('site').agg(circles=('circle_id','size'),
    median_cover_pct=('cover_pct','median'),min_cover_pct=('cover_pct','min'),
    max_cover_pct=('cover_pct','max'),min_roi_coverage=('roi_coverage_fraction','min')).round(3))
display(STAT.groupby(['site','fold']).agg(circles=('circle_id','size'),
    mean_cover_pct=('cover_pct','mean')).round(2))
STAT.to_csv(OUT/'01_reference_balance.csv',index=False)
fig,ax=plt.subplots(figsize=(9,4))
for ix,site in enumerate(SITES):
    vals=STAT.loc[STAT.site==site,'cover_pct'].to_numpy()
    ax.scatter(np.full(len(vals),ix)+np.linspace(-.12,.12,len(vals)),vals)
ax.set_xticks(range(len(SITES)),SITES);ax.set_ylabel('Exhaustive circle Musanga cover (%)')
ax.set_title('Within-site reference range (each point = one circle)')
ax.grid(axis='y',alpha=.2)
fig.savefig(OUT/'01_cover_by_site.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)


## 10 — Data contract and model-development hand-off

Notebook 02 should compare exhaustive-only and exhaustive-plus-partial supervision, using **the same folds**. Select pooled versus site-adapted U-Nets based on out-of-fold predictions, fraction bias and calibration, not only segmentation F1. Then refit on all labels for production probability maps **within ROI.shp only**. Include an independent valid RGB/nodata mask while reading TIFF windows.

The target rasterization function is part of this version-pinned notebook; do not redefine label precedence silently.


In [ ]:
contract=dict(schema='musanga_v4_reference_01_rgb_dsm_no_MOS34',input_manifest=str(manifest_path),
  site_groups=SITES,mosaics=list(MOSAICS),circle_count=len(CIRCLES),
  annotation_version='KVO_field_final',baseline_fold_manifest=str(V3_FOLDS),
  exploratory_annotation_reference=False,
  folds=FOLDS,seed=SEED,patch_px=PATCH,buffer_m=BUFFER_M,
  circle_reference_generation='KVO_field_final_30_no_MOS34',excluded_mosaic=EXCLUDED_MOSAIC,excluded_circle_ids=sorted(EXCLUDED_IDS),
  circle_ids=list(map(int,sorted(CIRCLES.circle_id))),
  exhaustive_all_polygons_musanga=True,
  output_directory=str(OUT),
  parent_v3_1_reference_manifest=str(manifest_path),
  v4_nb00_audit=str(V4_AUDIT),
  rgb_dsm_paths=DSM_PATHS,
  rgb_dsm_pairing_report=str(OUT/'01_rgb_dsm_pairing_metadata.csv'),
  rgb_dsm_circle_coverage_report=str(OUT/'01_dsm_circle_coverage_diagnostic.csv'),
  dsm_diagnostic_support='retained circle intersection ROI.shp intersection RGB dataset_mask',
  dsm_policy='SfM canopy surface; windowed RGB-grid WarpedVRT; no DTM/CHM; inspect crown co-registration',
  general_roi=str(ROI_FILE),exhaustive_roi=str(CIRCLE_FILE),
  exhaustive_positive=str(EXH_FILE),partial_crowns={s:str(p) for s,p in TREE_FILES.items()},
  label_policy=dict(
    exhaustive='positive polygons Musanga; remaining confirmed background non-Musanga',
    partial='labelled crowns only; blank unknown',unknown='weight zero',
    disagreement='explicit contradictions ignored during training, retained for full-reference validation',
    production='train with all usable annotations after spatial CV selection; never predict outside general ROI'),
  cv_policy='three within-forest spatial folds, MOS treated as one forest group; 15m RGB window exclusion',
  evaluation=['pixel AUC/AP/Brier','circle mean-probability fraction bias/MAE',
              'thresholded cover bias','spatial F1/IoU','site-macro scores'],
  files=['01_geometry_repairs.csv','01_mosaic_registry.csv','01_circle_spatial_folds.csv',
         '01_candidate_windows.csv','01_cv_window_manifest.csv',
         '01_reference_balance.csv','01_label_QC.png'])
(OUT/'01_data_contract.json').write_text(json.dumps(contract,indent=2))
print('Notebook 01 complete. Outputs:')
for f in sorted(OUT.glob('01_*')):print(' ',f.name)
print('IMPORTANT: V4 reference outputs isolated; NB02B must point to v4 and use same folds.')
print('No training, full-mosaic raster copying or inference performed.')


## 11 — New field plot registry (65 exhaustive plots; independent BA measurements)

The **old** stem-count `Plot_Data.shp` registry is superseded. The source is `Data/Field/Plot_Data_BA_nobuffer_ROI 2.gpkg`. This block provides a lightweight field-input overview; the authoritative repaired geometries, new training supervision and spatial holdouts are created in Sections 17–20. The field BA% is independent of the human crown delineation, but plots used to train a U-Net cannot simultaneously be called spatially held-out model tests.


In [ ]:
# NB01.11 — Register the new 65-plot independent field BA reference, no old shapefile.
PLOT_FILE=DATA/'Field'/'Plot_Data_BA_nobuffer_ROI 2.gpkg'
assert PLOT_FILE.is_file(),f'Field source missing: {PLOT_FILE}'
PLOTS=gpd.read_file(PLOT_FILE,layer='Plot_Data_BA_nobuffer_ROI')
assert len(PLOTS)==65 and PLOTS.crs is not None
assert {'Plot_name','pct_BA_musanga','surface_ha','Site'}.issubset(PLOTS.columns)
FIELD_PLOTS=PLOTS[['Plot_name','Site','surface_ha','pct_BA_musanga']].copy()
FIELD_PLOTS['polygon_area_ha']=PLOTS.to_crs('EPSG:32635').geometry.area.to_numpy()/10000
FIELD_PLOTS['geometry_area_mismatch']=~np.isclose(
    FIELD_PLOTS.polygon_area_ha,FIELD_PLOTS.surface_ha,rtol=.02,atol=.003)
FIELD_PLOTS.to_csv(OUT/'01_field65_input_registry.csv',index=False)
display(FIELD_PLOTS.groupby('Site').agg(
    n_plots=('Plot_name','size'),mean_field_BA_pct=('pct_BA_musanga','mean'),
    area_mismatch_n=('geometry_area_mismatch','sum')).round(2))
FIELD_CONTRACT=dict(file=str(PLOT_FILE),fraction_column='pct_BA_musanga',
    plot_count=int(len(FIELD_PLOTS)),data_type='field_DBH_based_relative_basal_area',
    note='See V4 NB00 Sections 06-08 and V4 NB01 Sections 17-20 for authoritative geometry/reference/folds')
(OUT/'01_external_field_plots_contract.json').write_text(json.dumps(FIELD_CONTRACT,indent=2))
print('New field registry:',len(FIELD_PLOTS),'plots; no old Plot_Data.shp used here.')


## 12 — Direct paired comparison: V3 versus V3_1 Musanga fraction

Uses the NB00 polygon-union audit (v1/v2 on exactly the same 34 circle boundaries), restricted to the original **30 retained circles**, mapped to the frozen V3 site and fold table. Reports mean and median per-circle cover as well as site-area-weighted cover (the latter is the area fraction across circles, provided circles do not overlap). This is reference annotation cover, **not** the independent field-plot measurements. No retraining or raster calculation.


In [ ]:
AUDIT_COMPARE=DATA.parent/'Outputs/V3_1/00_input_audit/00_circle_v1_v2_comparison.csv'
assert AUDIT_COMPARE.exists(),AUDIT_COMPARE
q=pd.read_csv(AUDIT_COMPARE)
required={'circle_id','musanga_cover_pct_v1','musanga_cover_pct_v2','musanga_union_area_m2_v1','musanga_union_area_m2_v2'}
assert required.issubset(q.columns),set(q.columns)
reference=LINK[['circle_id','site','fold']].merge(q,on='circle_id',validate='one_to_one')
assert len(reference)==30 and reference.circle_id.nunique()==30
circle_area=CIRCLES[['circle_id','geometry']].copy()
# Compute real area in metres even if circle layer uses geographic coordinates.
if not CIRCLES.crs.is_projected:
    circle_area=circle_area.set_geometry('geometry').to_crs(CIRCLES.estimate_utm_crs())
circle_area['area_m2']=circle_area.geometry.area
reference=reference.merge(circle_area[['circle_id','area_m2']],on='circle_id',validate='one_to_one')
reference['delta_cover_pp']=reference.musanga_cover_pct_v2-reference.musanga_cover_pct_v1
reference['cover_ratio_v2_v1']=reference.musanga_cover_pct_v2 / reference.musanga_cover_pct_v1.replace(0,np.nan)
summary=[]
for site,g in reference.groupby('site',sort=False):
    item=dict(site=site,circles=len(g))
    for ver in ('v1','v2'):
        c=g['musanga_cover_pct_'+ver]
        item[ver+'_mean_pct']=float(c.mean())
        item[ver+'_median_pct']=float(c.median())
        item[ver+'_area_weighted_pct']=float(100*g['musanga_union_area_m2_'+ver].sum()/g.area_m2.sum())
    item['delta_mean_pp']=item['v2_mean_pct']-item['v1_mean_pct']
    item['delta_median_pp']=item['v2_median_pct']-item['v1_median_pct']
    item['delta_area_weighted_pp']=item['v2_area_weighted_pct']-item['v1_area_weighted_pct']
    item['area_weighted_change_pct']=(100*(item['v2_area_weighted_pct']/item['v1_area_weighted_pct']-1)
          if item['v1_area_weighted_pct'] else np.nan)
    summary.append(item)
paired_sites=pd.DataFrame(summary)
print('=== V3 (v1) vs V3_1 (v2): MUSANGA COVER BY SITE ===')
display(paired_sites.round(3))
print('=== PAIRED CIRCLES: original V3 site and fold IDs ===')
display(reference[['site','fold','circle_id','musanga_cover_pct_v1',
                   'musanga_cover_pct_v2','delta_cover_pp']].sort_values(
                   ['site','circle_id']).round(3))
paired_sites.to_csv(OUT/'12_V3_vs_V3_1_site_cover_comparison.csv',index=False)
reference.to_csv(OUT/'12_V3_vs_V3_1_circle_cover_comparison.csv',index=False)
v3_balance=DATA.parent/'Outputs/V3/01_reference_data_no_MOS34/01_reference_balance.csv'
if v3_balance.exists():
    old=pd.read_csv(v3_balance)[['circle_id','cover_pct']].rename(columns={'cover_pct':'saved_V3_cover_pct'})
    check=reference.merge(old,on='circle_id',validate='one_to_one')
    check['audit_minus_saved_V3_pp']=check.musanga_cover_pct_v1-check.saved_V3_cover_pct
    max_abs=float(check.audit_minus_saved_V3_pp.abs().max())
    print('Audit v1 vs saved V3 baseline maximum difference (pp):',round(max_abs,4))
    if max_abs>0.1:
        print('WARNING: Audit-based v1 cover differs from saved V3 label balance.'
              ' Inspect geometry repairs and reference semantics before exact attribution.')
    check.to_csv(OUT/'12_baseline_v1_vs_saved_V3_QA.csv',index=False)
print('SAVED:',OUT/'12_V3_vs_V3_1_site_cover_comparison.csv')


## 13 — V4 DSM structural features: five small ROI-only QA windows (NEW)

**Purpose:** inspect local canopy-surface prominence, detrended roughness and curvature **before modifying V3_1 NB02B**. This uses only five small 512 × 512 RGB-native windows, one per active mosaic. No full-mosaic computation and no new training/evaluation split.

The DSM is an **SfM visible canopy surface**, not a ground-normalized canopy height model. Raw absolute elevations are not used as predictors. Missing DSM is never imputed as canopy height; the masks follow `ROI.shp`, valid RGB and valid DSM. A valid DSM pair still needs crown-scale visual registration QA. The displayed plots are QA, **not held-out model validation**.


In [ ]:
# V4 SECTION 13 — lightweight, ROI-masked DSM feature QA.
# No changes to V3_1 original training/ref/fold blocks.
from scipy.ndimage import gaussian_filter
from rasterio.windows import Window

QA_CHIP=512
QA_BROAD_SIGMA_M=2.5  # detrending scale; experimental, not optimized
QA_FINE_SIGMA_M=0.75  # texture scale; experimental, not optimized

def _masked_gaussian(a,valid,sigma_px):
    """Normalized convolution; never treat DSM NoData or outside ROI as elevation zero."""
    v=valid.astype('float32')
    numerator=gaussian_filter(np.where(valid,a,0).astype('float32'),sigma_px,mode='nearest')
    denominator=gaussian_filter(v,sigma_px,mode='nearest')
    out=numerator/np.maximum(denominator,1e-6)
    return out,denominator

def sfm_surface_features(z,valid,pixel_m):
    """Local SfM surface descriptors (not CHM), with strict valid support."""
    z=np.asarray(z,dtype='float32')
    valid=np.asarray(valid,dtype=bool)&np.isfinite(z)
    s_broad=max(1.,QA_BROAD_SIGMA_M/pixel_m)
    s_fine=max(1.,QA_FINE_SIGMA_M/pixel_m)
    base,base_weight=_masked_gaussian(z,valid,s_broad)
    local_mean,_=_masked_gaussian(z,valid,s_fine)
    local_second,_=_masked_gaussian(z*z,valid,s_fine)
    prominence=z-base
    roughness=np.sqrt(np.maximum(local_second-local_mean*local_mean,0))
    # Convexity: negative Laplacian of locally smoothed surface.
    # Units are m^-1, approximate continuous second spatial derivative.
    smoothed,_=_masked_gaussian(z,valid,s_fine)
    from scipy.ndimage import laplace
    convexity=-laplace(smoothed,mode='nearest')/(pixel_m**2)
    # Suppress boundary features where the kernel includes much invalid support.
    # This avoids artifact-driven peaks along DSM gaps and ROI boundaries.
    feature_valid=valid&(base_weight>.90)
    out=np.stack([prominence,roughness,convexity],axis=-1).astype('float32')
    out[~feature_valid]=np.nan
    return out,feature_valid

def _qa_crop_for_circle(name,circle_id,size=QA_CHIP):
    """Small RGB-native crop around a known circle, wholly within mosaic dimensions."""
    d=GEO[name]
    circle=CIRCLES.loc[CIRCLES.circle_id==circle_id].to_crs(d['crs']).geometry.iloc[0]
    rc=rasterio.transform.rowcol(d['transform'],circle.centroid.x,circle.centroid.y)
    row=max(0,min(d['height']-size,int(rc[0])-size//2))
    col=max(0,min(d['width']-size,int(rc[1])-size//2))
    return Window(col,row,size,size)

qa_metrics=[]
for name in MOSAICS:
    eligible=CIRCLE_DSM[(CIRCLE_DSM.mosaic==name)&(CIRCLE_DSM.dsm_nonzero_fraction_in_roi_rgb>=.95)]
    assert len(eligible)>0, f'No DSM-covered reference circle for {name}'
    circle_id=int(eligible.iloc[0].circle_id)
    win=_qa_crop_for_circle(name,circle_id)
    with rasterio.open(MOSAICS[name]['path']) as rgb_ds:
        raw=rgb_ds.read([1,2,3],window=win,masked=True)
        rgb=raw.filled(0).transpose(1,2,0).astype('float32')
        rgb_valid=(~np.any(np.ma.getmaskarray(raw),axis=0)) & (rgb_ds.dataset_mask(window=win)>0)
        # Preserve RGB semantics from V3_1; display only.
        rgb_display=np.clip(rgb/255.,0,1)
        pixel_m=np.sqrt(abs(rgb_ds.transform.a*rgb_ds.transform.e))
    roi=label_window(name,win,role='production')['roi'].astype(bool)
    z=dsm_window_on_rgb_grid(MOSAICS[name]['path'],DSM_PATHS[name],win)
    zz=np.asarray(z.data,dtype='float32')
    support=roi&rgb_valid&(~np.ma.getmaskarray(z))&np.isfinite(zz)&(~np.isclose(zz,0,atol=1e-6))
    feats,feature_valid=sfm_surface_features(zz,support,pixel_m)
    if not support.any():
        raise RuntimeError(f'No DSM within ROI at {name} circle {circle_id}')
    qa_metrics.append(dict(site=MOSAICS[name]['site'],mosaic=name,circle_id=circle_id,
        roi_rgb_pixels=int((roi&rgb_valid).sum()),
        dsm_supported_pixels=int(support.sum()),
        feature_pixels=int(feature_valid.sum()),
        sampled_dsm_pct=round(100*support.sum()/max((roi&rgb_valid).sum(),1),2),
        sampled_feature_pct=round(100*feature_valid.sum()/max((roi&rgb_valid).sum(),1),2),
        median_dsm_m=float(np.median(zz[support])),
        prominence_p95_m=float(np.nanpercentile(feats[...,0],95)) if feature_valid.any() else np.nan,
        roughness_p50_m=float(np.nanmedian(feats[...,1])) if feature_valid.any() else np.nan))
    fig,axes=plt.subplots(1,5,figsize=(18,4),constrained_layout=True)
    image=np.where((roi&rgb_valid)[...,None],rgb_display,np.nan)
    axes[0].imshow(image);axes[0].set_title(f'{name[:22]} RGB')
    dsm_plot=np.where(support,zz,np.nan)
    axes[1].imshow(dsm_plot,cmap='terrain');axes[1].set_title('SfM DSM elevation (m)')
    for ax,idx,title in zip(axes[2:],[0,1,2],
                             ['Local prominence (m)','Local roughness (m)','Local convexity (m⁻¹)']):
        ax.imshow(feats[...,idx],cmap='coolwarm' if idx!=1 else 'viridis')
        ax.set_title(title)
    for ax in axes:ax.set_xticks([]);ax.set_yticks([])
    fig.suptitle(f'V4 DSM QA — circle {circle_id}; inside ROI only',fontsize=12)
    fig.savefig(OUT/f'13_DSM_QA_{name.replace(".tif","").replace(" ","_")}.png',dpi=140)
    plt.show();plt.close(fig)
QA_DSM=pd.DataFrame(qa_metrics)
QA_DSM.to_csv(OUT/'13_dsm_feature_qa_summary.csv',index=False)
display(QA_DSM.round(3))
print('Five small ROI-masked QA crops complete. Review RGB-to-DSM crown alignment visually.')
print('These are exploratory SfM surface descriptors. Do not treat absolute DSM as height above ground.')


## 14 — MS coverage inside ROI.shp and frozen V3_1 circles (NEW)

Read the `NB00` multispectral registry; calculate **circle ∩ ROI.shp ∩ original RGB footprint ∩ each independent MS acquisition**. The first stage uses geometry and TIFF headers only. For circles with spatial overlap, read a small local 128 × 128 window in the **green band** (Baego: described Green band 2) and check per-pixel MS validity jointly with valid RGB. Do **not** require a Yangambi right-hand MS acquisition to coincide with its left-hand Carlier circles; report absent overlap as absent data, not poor predictions.

This is a **coverage audit only**, not multispectral calibration or co-registration validation. Two-band individual MS TIFFs may contain an alpha/validity band; inspect band descriptions and mask metadata before deciding which channels to train on. The duplicate MOS1–2 red-edge filenames are not used here. Never treat unobserved MS as zero reflectance.


In [ ]:
# Section 14: efficient MS spatial eligibility QA; no changes to V3_1 folds/labels.
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as affine_from_bounds
from rasterio.features import rasterize
from shapely.geometry import box

ms_csv=DATA.parent/'Outputs/v4/00_input_audit/00_ms_raster_headers.csv'
assert ms_csv.exists(),'Rerun V4 NB00 MS Section 04B to create its header registry'
ms=pd.read_csv(ms_csv)
required={'group','band_hint','name','path','bands','left','bottom','right','top','crs'}
assert required.issubset(ms.columns),required-set(ms.columns)
# Select exactly one *known* green source per independent acquisition; Baego is a multiband raster.
sources={}
for group,grp in ms.groupby('group'):
    if group=='Baego_MS':
        choices=grp[(grp.name=='Baego_MS_4band_RGB_aligned.tif')&(grp.bands==4)]
        band_index=2   # Verified from TIFF descriptions: Red, Green, RedEdge, NIR.
    else:
        choices=grp[grp.band_hint=='green']
        band_index=1   # Spectral band candidate in individual green TIFF; verify radiometry next.
    assert len(choices)==1,(group,choices[['name','path']].to_dict('records'))
    sources[group]=(choices.iloc[0],band_index)

MS_RGB={
 'Baego_MS':'Baego.tif',
 'MOS_1_2_MS':'Mos_1_2_transparent_mosaic_group1_georef.tif',
 'MOS_5_6_MS':'Mos_5_6_transparent_mosaic_group1_georef.tif',
 'Yangambi_YBJoseph_right_MS':'Yangambi_georef.tif',
 'Yangambi_YBForgeo_right_MS':'Yangambi_georef.tif'}
# Never infer that Yangambi Joseph/Forgeo acquisitions are Carlier/left.
# They are checked independently against the actual stored ROI and circle locations.
MS_GRID=128
rows=[]
for group,(info,band_idx) in sources.items():
    name=MS_RGB[group]
    roi_geom=roi_unions[name]  # Original V3_1 ROI.shp in RGB mosaic CRS
    with rasterio.open(MOSAICS[name]['path']) as rgb, rasterio.open(info.path) as ms_src:
        if ms_src.crs!=rgb.crs:
            raise AssertionError((group,ms_src.crs,rgb.crs))
        ms_footprint=box(*ms_src.bounds)
        for _,rec in LINK.loc[LINK.mosaic==name].iterrows():
            circle=CIRCLES.loc[CIRCLES.circle_id==int(rec.circle_id)].to_crs(rgb.crs).geometry.iloc[0]
            roi_circle=circle.intersection(roi_geom).intersection(box(*rgb.bounds))
            if roi_circle.is_empty:
                continue
            eligible=roi_circle.intersection(ms_footprint)
            fraction=float(eligible.area/roi_circle.area) if roi_circle.area else 0.
            result=dict(group=group,site=rec.site,mosaic=name,circle_id=int(rec.circle_id),
                fold=int(rec.fold),ms_name=info['name'],
                roi_circle_area_m2=float(roi_circle.area),
                ms_bbox_overlap_fraction_in_roi_circle=fraction,
                ms_valid_fraction_in_roi_rgb=np.nan,
                valid_joint_sample_pixels=0,
                roi_rgb_sample_pixels=0,
                sampled_green_p50=np.nan,
                ms_mask_flags=str(ms_src.mask_flag_enums[band_idx-1]))
            # No reading when there is no overlap; this is frequent for Yangambi.
            if fraction>0:
                tr=affine_from_bounds(*circle.bounds,MS_GRID,MS_GRID)
                support_geom=roi_circle
                support=rasterize([(support_geom,1)],out_shape=(MS_GRID,MS_GRID),
                    transform=tr,fill=0,dtype='uint8').astype(bool)
                with WarpedVRT(rgb,crs=rgb.crs,transform=tr,width=MS_GRID,height=MS_GRID,
                               resampling=Resampling.nearest) as rr:
                    rgb_ok=rr.dataset_mask()>0
                denominator=int((support&rgb_ok).sum())
                result['roi_rgb_sample_pixels']=denominator
                with WarpedVRT(ms_src,crs=rgb.crs,transform=tr,
                               width=MS_GRID,height=MS_GRID,
                               resampling=Resampling.nearest) as mm:
                    arr=mm.read(band_idx,masked=True)
                numeric=np.asarray(arr.data,dtype=np.float64)
                ms_ok=(~np.ma.getmaskarray(arr)) & np.isfinite(numeric) & support & rgb_ok
                result['valid_joint_sample_pixels']=int(ms_ok.sum())
                result['ms_valid_fraction_in_roi_rgb']=float(ms_ok.sum()/denominator) if denominator else np.nan
                if ms_ok.any():result['sampled_green_p50']=float(np.median(numeric[ms_ok]))
            rows.append(result)
MS_CIRCLES=pd.DataFrame(rows).sort_values(['site','group','circle_id'])
assert not MS_CIRCLES.empty,'No MS group/circle metadata produced'
MS_CIRCLES.to_csv(OUT/'14_ms_circle_coverage_diagnostic.csv',index=False)
print('=== MS coverage inside ROI ∩ RGB ∩ frozen reference circles ===')
display(MS_CIRCLES[['group','circle_id','fold',
    'ms_bbox_overlap_fraction_in_roi_circle','ms_valid_fraction_in_roi_rgb',
    'sampled_green_p50']].round(3))
summary=MS_CIRCLES.groupby('group').agg(
    evaluated_circles=('circle_id','count'),
    bbox_intersecting_circles=('ms_bbox_overlap_fraction_in_roi_circle',lambda x:int((x>0).sum())),
    sampled_valid_ge95=('ms_valid_fraction_in_roi_rgb',lambda x:int((x>=.95).sum())),
    min_nonmissing_valid=('ms_valid_fraction_in_roi_rgb','min'))
display(summary.round(3))
summary.to_csv(OUT/'14_ms_coverage_by_acquisition.csv')
print('No Yoko MS; no Carlier/left Yangambi MS.')
print('NEXT: evaluate all four spectral bands + radiometric calibration and image-level registration')
print('Only ROI-based reference support is eligible; no training inputs altered.')


## 15 — Four-band MS validity, scale and co-registration QA (V4 only)

**Run after Section 14.** For every MS-covered frozen reference circle, read small (64 × 64) windows of **Red, Green, RedEdge and NIR** on a common grid within `ROI.shp ∩ valid RGB`. Record all-band joint coverage, ranges, percentile statistics, and exploratory raw-band NDVI/NDRE. This is a **diagnostic**, not radiometric calibration; raw band ratios are only interpretable after checking physical scaling.

**Baego:** use the GeoTIFF's described `Red, Green, RedEdge, NIR` order. **Other acquisitions:** inspect individual band-1 data; band 2 is *not* assumed to be an independent spectral channel. **MOS1–2 duplicate red-edge:** sample both candidate TIFFs, report discrepancies, and require explicit resolution if they differ. No automatic approval of a mismatched duplicate.

At most one representative 256 × 256 visualization per acquisition: RGB (same spatial support), MS green, NIR, raw NDVI and red-edge ratio, with masked backgrounds. A consistent CRS/extent is not proof of subpixel crown co-registration; inspect spatial edges before training.

Outputs: `15_ms_fourband_circle_qa.csv`, `15_ms_acquisition_summary.csv`, `15_ms_duplicate_rededge_qa.csv`, `15_ms_band_registry.json`, and small QA PNGs. **V3_1 code, masks, fold IDs and exclusions are untouched.**

In [ ]:
# V4 Section 15 — all four MS bands on matched ROI / valid-RGB samples.
# Reuses MS_CIRCLES, LINK and ROI objects from the original V3_1 flow.
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as affine_from_bounds
from rasterio.features import rasterize
from pathlib import Path
from contextlib import ExitStack
import json
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt

MS_BANDS=('red','green','red_edge','nir')
MS_SAMPLE_PX=64
MS_FIG_PX=256
MS_EXPLICIT_RED_EDGE={}  # set {'MOS_1_2_MS': 'exact_filename.tif'} only if duplicate versions differ

def ms_single_band_file(group,band):
    choices=ms[(ms.group==group)&(ms.band_hint==band)].copy()
    if not len(choices):
        raise AssertionError(f'Missing {band} for {group}')
    if len(choices)==1:
        return str(choices.iloc[0].path),1
    if group!='MOS_1_2_MS' or band!='red_edge':
        raise AssertionError(f'Unexpected duplicate {group}/{band}: {choices.name.tolist()}')
    # Explicit file choice only. The duplicate comparison below determines whether
    # the other file agrees; do not silently choose an ambiguous one.
    specified=MS_EXPLICIT_RED_EDGE.get(group,'')
    if not specified:
        return None,None
    chosen=choices[choices.name==specified]
    assert len(chosen)==1,(specified,choices.name.tolist())
    return str(chosen.iloc[0].path),1

def ms_sources(group):
    if group=='Baego_MS':
        choices=ms[(ms.group==group)&(ms.name=='Baego_MS_4band_RGB_aligned.tif')]
        assert len(choices)==1
        path=str(choices.iloc[0].path)
        with rasterio.open(path) as ds:
            desc=tuple((v or '').lower().replace(' ','').replace('_','') for v in ds.descriptions)
            assert desc==('red','green','rededge','nir'),('Baego MS descriptions changed',ds.descriptions)
        return {'red':(path,1),'green':(path,2),'red_edge':(path,3),'nir':(path,4)}
    return {b:ms_single_band_file(group,b) for b in MS_BANDS}

def read_ms_samples(group,rec,sz,band_files):
    name=MS_RGB[group]
    with rasterio.open(MOSAICS[name]['path']) as rgb:
        circle=CIRCLES.loc[CIRCLES.circle_id==int(rec.circle_id)].to_crs(rgb.crs).geometry.iloc[0]
        roi_circle=circle.intersection(roi_unions[name]).intersection(box(*rgb.bounds))
        if roi_circle.is_empty: return None
        tr=affine_from_bounds(*circle.bounds,sz,sz)
        footprint=rasterize([(roi_circle,1)],out_shape=(sz,sz),
             transform=tr,dtype='uint8',fill=0).astype(bool)
        with WarpedVRT(rgb,crs=rgb.crs,transform=tr,width=sz,height=sz,
                       resampling=Resampling.nearest) as rgbv:
            valid_rgb=rgbv.dataset_mask()>0
            raw_rgb=rgbv.read([1,2,3],masked=True)
        support=footprint&valid_rgb
        if not support.any():return None
        arrays={}
        band_masks={}
        with ExitStack() as stack:
            for band,(path,idx) in band_files.items():
                ds=stack.enter_context(rasterio.open(path))
                assert ds.crs==rgb.crs,(group,band,ds.crs,rgb.crs)
                vrt=stack.enter_context(WarpedVRT(ds,crs=rgb.crs,transform=tr,
                                 width=sz,height=sz,resampling=Resampling.nearest))
                sample=vrt.read(idx,masked=True)
                values=np.asarray(sample.data,dtype='float64')
                valid=(~np.ma.getmaskarray(sample))&np.isfinite(values)
                arrays[band]=values
                band_masks[band]=valid&support
        joint=support.copy()
        for mask in band_masks.values():joint &= mask
        return dict(support=support,joint=joint,arrays=arrays,
                    masks=band_masks,rgb=raw_rgb.filled(0).transpose(1,2,0).astype('float32'),
                    transform=tr)

# For MOS1–2 duplicate red-edge, compare the two source TIFFs using identical
# small samples on the six existing MS-covered circles. Do not load entire rasters.
duplicate_rows=[]
choices=ms[(ms.group=='MOS_1_2_MS')&(ms.band_hint=='red_edge')]
if len(choices)==2:
    p1,p2=map(str,choices.path)
    for cid in MS_CIRCLES.loc[(MS_CIRCLES.group=='MOS_1_2_MS') &
                              (MS_CIRCLES.ms_bbox_overlap_fraction_in_roi_circle>0),'circle_id']:
        record=LINK.loc[LINK.circle_id==int(cid)].iloc[0]
        name=MS_RGB['MOS_1_2_MS']
        with rasterio.open(MOSAICS[name]['path']) as rgb:
            circle=CIRCLES.loc[CIRCLES.circle_id==int(cid)].to_crs(rgb.crs).geometry.iloc[0]
            tr=affine_from_bounds(*circle.bounds,MS_SAMPLE_PX,MS_SAMPLE_PX)
            roi_circle=circle.intersection(roi_unions[name]).intersection(box(*rgb.bounds))
            footprint=rasterize([(roi_circle,1)],out_shape=(MS_SAMPLE_PX,MS_SAMPLE_PX),
                                 transform=tr,dtype='uint8',fill=0).astype(bool)
        candidates=[]
        for p in (p1,p2):
            with rasterio.open(p) as ds, WarpedVRT(ds,crs=rgb.crs,transform=tr,
                 width=MS_SAMPLE_PX,height=MS_SAMPLE_PX,
                 resampling=Resampling.nearest) as vrt:
                v=vrt.read(1,masked=True)
                candidates.append(v)
        a,b=candidates
        aok=~np.ma.getmaskarray(a);bok=~np.ma.getmaskarray(b)
        common=footprint&aok&bok
        difference=np.abs(np.asarray(a.data,dtype='float64')-
                          np.asarray(b.data,dtype='float64'))
        duplicate_rows.append(dict(circle_id=int(cid),file_a=Path(p1).name,file_b=Path(p2).name,
            common_sample_pixels=int(common.sum()),
            mask_disagreement_pixels=int((footprint&(aok^bok)).sum()),
            max_abs_difference=float(difference[common].max()) if common.any() else np.nan))
DUPLICATE_MS=pd.DataFrame(duplicate_rows)
DUPLICATE_MS.to_csv(OUT/'15_ms_duplicate_rededge_qa.csv',index=False)
print('=== MOS 1–2 duplicate red-edge QA ===')
display(DUPLICATE_MS)
if len(choices)==2 and not MS_EXPLICIT_RED_EDGE:
    # Only auto-identify duplicate as interchangeable if sampled masks AND
    # pixels agree in every tested ROI circle; still report that this is a sample audit.
    equivalent=bool(len(DUPLICATE_MS)>0 and
        (DUPLICATE_MS.common_sample_pixels>0).all() and
        (DUPLICATE_MS.mask_disagreement_pixels==0).all() and
        (DUPLICATE_MS.max_abs_difference.fillna(np.inf)<=1e-6).all())
    if equivalent:
        selected=sorted(choices.name,key=lambda s:('(2)' in s.lower(),len(s),s))[0]
        MS_EXPLICIT_RED_EDGE['MOS_1_2_MS']=selected
        print('Duplicate matched in sampled ROI circles; provisionally selected:',selected)
    else:
        print('Duplicate files DIFFER or are unverified. MOS1–2 four-band QA is held.')
        print('Review the two sources and set MS_EXPLICIT_RED_EDGE explicitly; rerun Section 15.')

source_registry={}
for group in sorted(sources):
    bands=ms_sources(group)
    if all(p is not None for p,i in bands.values()):
        source_registry[group]=bands
    else:
        print('SKIP unresolved four-band group:',group)
rows=[]
for group,files in source_registry.items():
    eligible=MS_CIRCLES[(MS_CIRCLES.group==group)&
                        (MS_CIRCLES.ms_bbox_overlap_fraction_in_roi_circle>0)]
    for _,rec in eligible.iterrows():
        sample=read_ms_samples(group,rec,MS_SAMPLE_PX,files)
        if sample is None:continue
        sup=sample['support'];joint=sample['joint'];vals=sample['arrays']
        item=dict(group=group,site=rec.site,circle_id=int(rec.circle_id),
            fold=int(rec.fold),n_roi_rgb=int(sup.sum()),n_joint=int(joint.sum()),
            joint_valid_fraction=float(joint.sum()/sup.sum()))
        for band in MS_BANDS:
            mask=sample['masks'][band]
            item[f'{band}_valid_frac']=float(mask.sum()/sup.sum())
            item[f'{band}_p01']=float(np.percentile(vals[band][mask],1)) if mask.any() else np.nan
            item[f'{band}_p50']=float(np.median(vals[band][mask])) if mask.any() else np.nan
            item[f'{band}_p99']=float(np.percentile(vals[band][mask],99)) if mask.any() else np.nan
        if joint.any():
            # Raw-ratio exploratory diagnostics only; calibrated reflectance NOT established.
            rd,ni,re=vals['red'][joint],vals['nir'][joint],vals['red_edge'][joint]
            ndvi=(ni-rd)/np.where(np.abs(ni+rd)>1e-9,ni+rd,np.nan)
            ndre=(ni-re)/np.where(np.abs(ni+re)>1e-9,ni+re,np.nan)
            item['raw_ndvi_p50']=float(np.nanmedian(ndvi))
            item['raw_ndre_p50']=float(np.nanmedian(ndre))
        rows.append(item)
MS_FOURBAND=pd.DataFrame(rows)
assert len(MS_FOURBAND)>0,'No valid four-band ROI samples'
MS_FOURBAND.to_csv(OUT/'15_ms_fourband_circle_qa.csv',index=False)
print('=== FOUR-BAND VALIDITY BY ROI-MATCHED CIRCLE ===')
display(MS_FOURBAND[['group','circle_id','fold','joint_valid_fraction',
    'red_p50','green_p50','red_edge_p50','nir_p50','raw_ndvi_p50','raw_ndre_p50']].round(3))
summary=MS_FOURBAND.groupby('group').agg(
    covered_circles=('circle_id','count'),
    allband_ge95=('joint_valid_fraction',lambda x:int((x>=.95).sum())),
    min_joint_valid=('joint_valid_fraction','min'),
    green_median=('green_p50','median'),nir_median=('nir_p50','median'),
    raw_ndvi_median=('raw_ndvi_p50','median'))
summary.to_csv(OUT/'15_ms_acquisition_summary.csv')
display(summary.round(3))
# One SMALL visualization per acquisition (inside ROI and valid RGB + all MS).
for group,files in source_registry.items():
    eligible=MS_FOURBAND[MS_FOURBAND.group==group].sort_values('joint_valid_fraction',ascending=False)
    if eligible.empty:continue
    cid=int(eligible.iloc[0].circle_id)
    rec=LINK.loc[LINK.circle_id==cid].iloc[0]
    q=read_ms_samples(group,rec,MS_FIG_PX,files)
    if q is None:continue
    joint=q['joint'];bands=q['arrays'];rgb=np.clip(q['rgb']/255.,0,1)
    red=bands['red'];nir=bands['nir'];re=bands['red_edge']
    with np.errstate(divide='ignore',invalid='ignore'):
        ndvi=np.where(np.abs(nir+red)>1e-9,(nir-red)/(nir+red),np.nan)
        ndre=np.where(np.abs(nir+re)>1e-9,(nir-re)/(nir+re),np.nan)
    panels=[np.where(q['support'][...,None],rgb,np.nan),
            np.where(joint,bands['green'],np.nan),
            np.where(joint,nir,np.nan),np.where(joint,ndvi,np.nan),
            np.where(joint,ndre,np.nan)]
    titles=['RGB','MS green (raw)','MS NIR (raw)','Raw NDVI*','Raw NDRE*']
    fig,axes=plt.subplots(1,5,figsize=(17,3.5),constrained_layout=True)
    for ax,arr,title in zip(axes,panels,titles):
        ax.imshow(arr,cmap=None if arr.ndim==3 else 'viridis')
        ax.set_title(title);ax.axis('off')
    fig.suptitle(f'{group}, circle {cid} — ROI-only, RGB/MS QA; *uncalibrated ratios',fontsize=10)
    fig.savefig(OUT/f'15_ms_qa_{group}.png',dpi=130)
    plt.show();plt.close(fig)
registry={'schema':'v4_ms_fourband_qa_1','bands':MS_BANDS,
          'sources':{g:{b:dict(path=p,band_index=i) for b,(p,i) in files.items()}
                     for g,files in source_registry.items()},
          'duplicate_red_edge_override':MS_EXPLICIT_RED_EDGE,
          'radiometric_calibration_verified':False,
          'crown_level_coregistration_verified':False,
          'support':'circle ∩ ROI.shp ∩ valid RGB ∩ all four MS bands',
          'interpretation':'NDVI/NDRE are exploratory raw-DN ratios, not calibrated reflectance'}
(OUT/'15_ms_band_registry.json').write_text(json.dumps(registry,indent=2))
print('Saved all four-band circle QA, summary, duplicate check, source registry and quicklook PNGs.')
print('Only within-ROI small samples were read; V3_1 labels/splits remain unchanged.')


## 16 — Targeted MS radiometry fault check (MOS 5–6 red; all-site red-edge/NIR)

The earlier four-band audit showed a red median of exactly zero for all three MOS5–6 circles and raw NDVI=1; this is **not an acceptable four-band validation signal** even if the TIFF mask reports valid. Diagnose **both bands** in the problematic red TIFF separately, and compare the spectral band-1 zero fractions in each MS acquisition. Work only inside `ROI.shp ∩ valid RGB ∩ reference circle`. Do not auto-switch to band 2, assume it is reflectance, or change training folds. Negative raw NDRE elsewhere is a **radiometric QA flag**, not proof of mislabeled bands.

In [ ]:
# V4 Section 16: SMALL ROI-contained MS DN/zero diagnostics. No model changes.
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as affine_from_bounds
from rasterio.features import rasterize
import numpy as np, pandas as pd, rasterio
MS_RAD_GRID=64
rad_rows=[]
for group,srcs in source_registry.items():
    ref_name=MS_RGB[group]
    covered=MS_CIRCLES.loc[(MS_CIRCLES.group==group)&
            (MS_CIRCLES.ms_bbox_overlap_fraction_in_roi_circle>0)]
    for _,r in covered.iterrows():
        cid=int(r.circle_id)
        with rasterio.open(MOSAICS[ref_name]['path']) as rgb:
            geom=CIRCLES.loc[CIRCLES.circle_id==cid].to_crs(rgb.crs).geometry.iloc[0]
            eligible=geom.intersection(roi_unions[ref_name]).intersection(box(*rgb.bounds))
            tr=affine_from_bounds(*geom.bounds,MS_RAD_GRID,MS_RAD_GRID)
            support=rasterize([(eligible,1)],out_shape=(MS_RAD_GRID,MS_RAD_GRID),
                  transform=tr,fill=0,dtype='uint8').astype(bool)
            with WarpedVRT(rgb,crs=rgb.crs,transform=tr,width=MS_RAD_GRID,
                    height=MS_RAD_GRID,resampling=Resampling.nearest) as rr:
                support &= rr.dataset_mask()>0
        for band,(src_path,band_index) in srcs.items():
            with rasterio.open(src_path) as ds:
                # Inspect both bands specifically for MOS5–6 red; no assumptions
                indexes=range(1,ds.count+1) if group=='MOS_5_6_MS' and band=='red' else (band_index,)
                for idx in indexes:
                    with WarpedVRT(ds,crs=ds.crs,transform=tr,width=MS_RAD_GRID,
                                   height=MS_RAD_GRID,resampling=Resampling.nearest) as aligned:
                        arr=aligned.read(idx,masked=True)
                    values=np.asarray(arr.data,dtype='float64')
                    valid=support & (~np.ma.getmaskarray(arr)) & np.isfinite(values)
                    vv=values[valid]
                    rad_rows.append(dict(group=group,circle_id=cid,band=band,source_band=idx,
                        file=Path(src_path).name,source_description=str(ds.descriptions[idx-1]),
                        source_mask_flags=str(ds.mask_flag_enums[idx-1]),source_nodata=ds.nodata,
                        n_support=int(support.sum()),n_valid=int(valid.sum()),
                        valid_fraction=float(valid.sum()/support.sum()) if support.any() else np.nan,
                        zero_fraction=float(np.isclose(vv,0,atol=1e-8).mean()) if len(vv) else np.nan,
                        p01=float(np.percentile(vv,1)) if len(vv) else np.nan,
                        p50=float(np.median(vv)) if len(vv) else np.nan,
                        p99=float(np.percentile(vv,99)) if len(vv) else np.nan))
RAD_DIAG=pd.DataFrame(rad_rows)
RAD_DIAG.to_csv(OUT/'16_ms_raw_band_diagnostics.csv',index=False)
print('=== MOS 5–6 RED: BOTH TIFF BANDS ===')
display(RAD_DIAG[(RAD_DIAG.group=='MOS_5_6_MS')&(RAD_DIAG.band=='red')][
    ['circle_id','source_band','source_description','source_mask_flags','source_nodata',
     'valid_fraction','zero_fraction','p01','p50','p99']].round(3))
print('=== ALL ACQUISITIONS: SOURCE SPECTRAL BANDS, ZERO FRACTION AND MEDIANS ===')
display(RAD_DIAG.groupby(['group','band','source_band']).agg(
    circles=('circle_id','count'),max_zero_fraction=('zero_fraction','max'),
    median_dn=('p50','median')).round(3))
print('Diagnostic only: do not use MOS5–6 raw NDVI=1 as a vegetation observation.')


## 17 — V4 field65 reference: exhaustive 65 plots plus 34 KVO circles and partial site crowns
Load the **NB00 audited** 65 plot geometries (not the original enlarged polygons) and the new labelled crowns. Nadine confirmed the **plots are exhaustively annotated**, with `Class_ID == 1` Musanga; other numeric classes are explicitly non-Musanga. Retain all valid repaired polygons, and audit source overlap. Within the exhaustive plots, missing positive Musanga crowns are negative for binary pixel segmentation **only when the field footprint and annotation completeness are confirmed**. The 34 KVO-corrected circles retain the previously established convention: every final crown polygon is Musanga, all other circle pixels are negative. Partial site crowns provide labelled positives/known negatives only, never a negative background. This dataset **does not inherit** the earlier 30-circle/MOS3–4 restriction unless a specific orthomosaic fails coverage QA.

In [ ]:
# NB01.17 — Authoritative V4 field reference, local geometry repair and source QA.
FIELD65_MANIFEST=DATA.parent/'Outputs/v4/00_input_audit/00_v4_field65_contract.json'
assert FIELD65_MANIFEST.exists(), 'Run V4 NB00 Sections 06-08 first'
REF65=json.loads(FIELD65_MANIFEST.read_text())
assert REF65['schema']=='musanga_v4_field65_input_audit_2026_11'
V4_REF_DIR=DATA.parent/'Outputs/v4/01_reference_field65_combined'
V4_REF_DIR.mkdir(parents=True,exist_ok=True)
P65=gpd.read_file(REF65['field_support_qa']).to_crs('EPSG:32635')
assert len(P65)==65 and P65.Plot_name.is_unique
NEW_CROWN_RAW=gpd.read_file(REF65['exhaustive_plot_crowns']).to_crs('EPSG:32635')
assert 'Class_ID' in NEW_CROWN_RAW.columns
NEW_CROWN_RAW['is_musanga']=pd.to_numeric(NEW_CROWN_RAW.Class_ID,errors='coerce').eq(1).astype('int8')
# Reuse NB00's area-conservative repair policy (<=5%). The original files remain unchanged.
# New plot crowns: use explicit <=5% repair tolerance (old load_polygons uses 1% for non-site files).
new_rows=[];new_geoms=[];new_qa=[]
for i,g in enumerate(NEW_CROWN_RAW.geometry):
    if g is None or g.is_empty:
        new_qa.append((i,'empty',np.nan));continue
    try:
        fixed=g if g.is_valid else unary_union(parts(make_valid(g)))
        if fixed is None or fixed.is_empty or not fixed.is_valid:
            new_qa.append((i,'quarantined',np.nan));continue
        delta=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-9)
        if delta>0.05:
            new_qa.append((i,'quarantined_gt5pct',delta));continue
        new_rows.append(i);new_geoms.append(fixed)
        new_qa.append((i,'valid' if g.is_valid else 'repaired',delta))
    except Exception:
        new_qa.append((i,'quarantined_exception',np.nan))
NEW_CROWN=NEW_CROWN_RAW.iloc[new_rows].copy()
NEW_CROWN.geometry=new_geoms
pd.DataFrame(new_qa,columns=['source_row','status','relative_area_change']).to_csv(
    V4_REF_DIR/'01_field65_crown_geometry_repairs.csv',index=False)
print('New exhaustive crown geometry status:')
display(pd.Series([q[1] for q in new_qa]).value_counts())

NEW_CROWN['is_musanga']=pd.to_numeric(NEW_CROWN.Class_ID,errors='coerce').eq(1).astype('int8')
assert len(NEW_CROWN)>0
assert NEW_CROWN['Class_ID'].notna().all()
P65['source_id']=P65.Plot_name.astype(str)
print('Field plot reference:',len(P65),'plots; annotations:',len(NEW_CROWN),
      'Musanga:',int(NEW_CROWN.is_musanga.sum()))
print('Class counts:');display(NEW_CROWN.Class_ID.value_counts(dropna=False))
print('Plot counts by site:');display(P65.groupby('Site').size())
print('Area-support mismatches flagged:',int(P65.area_mismatch.sum()))
# Separate native 34 circle boundaries from previous 30-circle filtered evaluation.
C34=gpd.read_file(REF65['circle_complete_roi']).to_crs('EPSG:32635')
C34['circle_id']=np.arange(len(C34))
assert len(C34)==34
E34=gpd.read_file(REF65['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(E34)==635
# Geometry repair in place; keep all successfully repaired labels.
def geoms_fixed(gdf,source,threshold=.05):
    good=[];index=[];qa=[]
    for i,g in enumerate(gdf.geometry):
        if g is None or g.is_empty:
            qa.append((source,i,'empty',np.nan));continue
        try:
            v=g if g.is_valid else make_valid(g)
            polys=parts(v)
            fixed=unary_union(polys) if polys else None
            if fixed is None or fixed.is_empty or not fixed.is_valid:
                qa.append((source,i,'quarantine',np.nan));continue
            change=abs(g.area-fixed.area)/max(g.area,fixed.area,1e-9)
            if change>threshold:
                qa.append((source,i,'quarantine_area_change',change));continue
            good.append(fixed);index.append(i)
            qa.append((source,i,'ok' if g.is_valid else 'repaired',change))
        except Exception:
            qa.append((source,i,'quarantine_exception',np.nan))
    o=gdf.iloc[index].copy();o.geometry=good
    return o,qa
C34,cqa=geoms_fixed(C34,'circle_roi',.05)
E34,eqa=geoms_fixed(E34,'kvo_circle_crowns',.05)
pd.DataFrame(cqa+eqa,columns=['source','row','status','area_change']).to_csv(
    V4_REF_DIR/'01_circle_geometry_repairs.csv',index=False)
assert len(C34)==34, 'Circle boundary quarantine: review before training'
# Flag, do not silently assert the same exact polygon count after geometry repair.
print('KVO circle crowns retained:',len(E34),'of 635')
PARTIAL={site:table.to_crs('EPSG:32635') for site,table in TREES.items()}
print('Four partial site annotation sources retained:',list(PARTIAL))
# V4 restart includes MOS 3-4; legacy V3_1 CACHE above excluded that mosaic.
# Register it exclusively for the new V4 field65 masks and candidate windows.
for info in A['mosaics']:
    if info['file'] in CACHE: continue
    if info['file'] != EXCLUDED_MOSAIC: continue
    with rasterio.open(info['path']) as ds:
        assert ds.crs and ds.crs.is_projected and ds.count>=3
        new_roi=ROI.to_crs(ds.crs)
        CACHE[info['file']]=dict(
            transform=ds.transform,crs=ds.crs,width=ds.width,height=ds.height,
            footprint=box(*ds.bounds),roi=new_roi,site=info['site'],
            trees=PARTIAL[info['site']].to_crs(ds.crs))
        print('V4 activated additional mosaic:',info['file'])
assert len(CACHE)==6,('Expected all six V3_1 RGB mosaics for V4; registered',list(CACHE))



## 18 — Joint spatial blocks and leave-block-out safeguards
Partition the 65 plots into **three contiguous within-site spatial strips** (PCA of plot centres), preserving entire plot identities, not pixels. Each circle is assigned to the fold of its nearest same-site plot (where site labels can be matched) and any circle overlapping another fold's plot is flagged for stricter exclusion; 15 m buffer around all held-out plot and circle supports prevents a training tile from reading its held-out neighborhood. Store folds **before** sampling imagery. The three-fold analysis measures spatial transfer within sites, not leave-one-forest-out transfer; evaluate LOSO separately. These are new V4 folds and must not be mixed with V3_1 scores.

In [ ]:
# NB01.18 — Spatially contiguous within-site plot blocks
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
P65['site_key']=P65.Site.astype(str).replace({'Mos':'MOS'})
SITES65=sorted(P65.site_key.unique())
FOLDS65=3
P65['cv_fold']=-1
for site,g in P65.groupby('site_key'):
    xy=np.column_stack([g.geometry.centroid.x,g.geometry.centroid.y])
    assert len(g)>=FOLDS65,(site,len(g))
    axis=PCA(n_components=1).fit_transform(xy).ravel()
    order=np.argsort(axis)
    # Contiguous, approximately equally sized geographic blocks
    groups=np.array_split(order,FOLDS65)
    for fold,positions in enumerate(groups):
        P65.loc[g.index[positions],'cv_fold']=fold
assert P65.cv_fold.ge(0).all()
display(P65.groupby(['site_key','cv_fold']).size().rename('n_plots').reset_index())
# Circle ownership: use original site assignment where available from the V3_1 audit.
# Assign by proximity to same-site plot centroids when explicit circle site is absent.
def site_from_circle(g):
    # Use the original V3_1 circle-mosaic assignment by original index.
    return None
C34['site_key']=None
if 'LINK' in globals() and {'circle_id','site'}.issubset(LINK.columns):
    mapping=LINK[['circle_id','site']].drop_duplicates().set_index('circle_id').site.to_dict()
    C34['site_key']=C34.circle_id.map(mapping)
# Fallback: smallest distance to the nearest plot of each site; flag for QA.
C34['site_inferred']=C34.site_key.isna()
centers=np.column_stack([P65.geometry.centroid.x,P65.geometry.centroid.y])
for idx,row in C34[C34.site_key.isna()].iterrows():
    dist=cdist([[row.geometry.centroid.x,row.geometry.centroid.y]],centers).ravel()
    C34.at[idx,'site_key']=P65.iloc[int(np.argmin(dist))].site_key
C34['cv_fold']=-1
C34['nearest_plot']=''
for i,row in C34.iterrows():
    subset=P65[P65.site_key==row.site_key]
    q=np.column_stack([subset.geometry.centroid.x,subset.geometry.centroid.y])
    di=cdist([[row.geometry.centroid.x,row.geometry.centroid.y]],q).ravel()
    nearest=subset.iloc[int(np.argmin(di))]
    C34.at[i,'nearest_plot']=nearest.Plot_name
    C34.at[i,'cv_fold']=int(nearest.cv_fold)
C34['cross_fold_plot_intersection']=False
for i,row in C34.iterrows():
    touching=P65[P65.geometry.intersects(row.geometry.buffer(15))]
    if len(touching) and touching.cv_fold.nunique()>1:
        C34.at[i,'cross_fold_plot_intersection']=True
    elif len(touching) and any(touching.cv_fold!=row.cv_fold):
        C34.at[i,'cross_fold_plot_intersection']=True
print('Circle-site inference needed for',int(C34.site_inferred.sum()),
      'circles; crossing >1 fold within 15m:',int(C34.cross_fold_plot_intersection.sum()))
# Crossfold-ambiguous circles are protected as exclusion buffers for ALL folds.
# They are not primary circle-evaluation reference until manually reviewed.
plot_columns=['source_id','site_key','cv_fold','pct_BA_musanga','area_mismatch','validation_geom_source']
P65[plot_columns].to_csv(V4_REF_DIR/'01_field65_plot_folds.csv',index=False)
C34[['circle_id','site_key','cv_fold','nearest_plot','site_inferred',
     'cross_fold_plot_intersection']].to_csv(V4_REF_DIR/'01_field65_circle_folds.csv',index=False)
fold_map=P65[['source_id','site_key','cv_fold']].copy()
display(fold_map.head())


## 18A — Repeatability audit: plot annotations versus corrected circle annotations, and **joint spatial folds**

**Before merging labels**, compare the two independently digitized Musanga masks only inside `plot ∩ circle`. Quantify cover difference, mask IoU and F1, and individual crown matching (greedy one-to-one, IoU ≥ 0.30). Export exact shared supports and disagreement polygons to GeoPackage for ArcGIS. The circle file is **KVO-corrected**: this assesses *plot annotation vs final circle-reference agreement*, not pure unmodified intra-annotator repeatability.

**Fold safety:** construct connected components of spatially overlapping plot and circle references. Each connected component must receive one CV fold; the 15-m holdout buffer remains enforced for windows and partial crowns. Spatial blocks are assigned by within-site position of each component, not by independent plot versus circle identities. Overlaps are diagnostic, **not additional independent training samples**.

In [ ]:
# NB01.18A — plot/circle disagreement and overlap-component spatial CV
from shapely.geometry import GeometryCollection
from shapely.ops import unary_union
from shapely import make_valid, set_precision
from scipy.spatial.distance import cdist
import scipy.stats as st

OVERLAP_DIR=V4_REF_DIR/'18A_plot_circle_consistency'
OVERLAP_DIR.mkdir(parents=True,exist_ok=True)
AREA_EPS=0.01 # m2: ignore point/line contacts
CROWNS_POS=NEW_CROWN.loc[NEW_CROWN.is_musanga==1].geometry.to_list()
CIRCLE_POS=E34.geometry.to_list()
PLOT_CROWN_TREE=gpd.GeoSeries(CROWNS_POS,crs=P65.crs).sindex
CIRCLE_CROWN_TREE=gpd.GeoSeries(CIRCLE_POS,crs=C34.crs).sindex

def poly_union_safe(items):
    """Conservative polygon-only topology; fails with context if unrecoverable."""
    items=[g for g in items if g is not None and not g.is_empty and g.area>0]
    if not items:return GeometryCollection()
    try:
        u=unary_union(items)
        return u if u.is_valid else make_valid(u)
    except Exception:
        u=unary_union([set_precision(make_valid(g),grid_size=0.01) for g in items])
        return make_valid(u)

def overlay_safe(a,b,op):
    if a is None or b is None or a.is_empty or b.is_empty:
        return GeometryCollection()
    try:
        ans=getattr(a,op)(b)
    except Exception:
        aa=set_precision(make_valid(a),grid_size=0.01)
        bb=set_precision(make_valid(b),grid_size=0.01)
        ans=getattr(aa,op)(bb)
    if not ans.is_valid:ans=make_valid(ans)
    return ans

def clipped_crowns(geoms,index,geom):
    # sindex is built for the same full geometry ordering as geoms
    chosen=list(index.query(geom,predicate='intersects'))
    return [overlay_safe(geoms[int(j)],geom,'intersection')
            for j in chosen if overlay_safe(geoms[int(j)],geom,'intersection').area>AREA_EPS]

pairs=[]; supports=[];diffs=[];crown_matches=[];bad=[]
for plot in P65.itertuples(index=False):
    for circle in C34.itertuples(index=False):
        try:
            common=overlay_safe(plot.geometry,circle.geometry,'intersection')
            if common.area<=AREA_EPS:continue
            pc=clipped_crowns(CROWNS_POS,PLOT_CROWN_TREE,common)
            cc=clipped_crowns(CIRCLE_POS,CIRCLE_CROWN_TREE,common)
            p_mask=poly_union_safe(pc); c_mask=poly_union_safe(cc)
            shared=overlay_safe(p_mask,c_mask,'intersection')
            a=float(common.area)
            ap=float(p_mask.area);ac=float(c_mask.area);ai=float(shared.area)
            au=max(0.,ap+ac-ai)
            disagreement=overlay_safe(p_mask,c_mask,'symmetric_difference')
            disagreement=unary_union(parts(disagreement)) if parts(disagreement) else GeometryCollection()
            plot_cover=100*ap/a
            circle_cover=100*ac/a
            # Greedy one-to-one matching of individual crowns over identical support.
            candidates=[]
            for ii,pg in enumerate(pc):
                for jj,cg in enumerate(cc):
                    if not pg.intersects(cg):continue
                    inter=float(overlay_safe(pg,cg,'intersection').area)
                    den=float(pg.area+cg.area-inter)
                    score=inter/den if den>0 else 0.
                    if score>=0.30:candidates.append((score,ii,jj))
            used_p=set();used_c=set();accepted=[]
            for score,ii,jj in sorted(candidates,reverse=True):
                if ii in used_p or jj in used_c:continue
                used_p.add(ii);used_c.add(jj);accepted.append((ii,jj,score))
            pair_id=f'{plot.Plot_name}__circle_{int(circle.circle_id)}'
            pairs.append(dict(pair_id=pair_id,Plot_name=str(plot.Plot_name),
                site=str(plot.site_key),circle_id=int(circle.circle_id),
                shared_area_m2=a,shared_fraction_of_plot=a/plot.geometry.area,
                plot_cover_pct=plot_cover,circle_cover_pct=circle_cover,
                cover_difference_pp=plot_cover-circle_cover,
                musanga_intersection_m2=ai,musanga_union_m2=au,
                mask_iou=ai/au if au>0 else 1.0,
                mask_f1=(2*ai/(ap+ac)) if ap+ac>0 else 1.0,
                disagreement_area_m2=float(disagreement.area),
                n_plot_crowns=len(pc),n_circle_crowns=len(cc),
                n_matched_crowns=len(accepted),
                crown_match_precision=len(accepted)/len(cc) if len(cc) else np.nan,
                crown_match_recall=len(accepted)/len(pc) if len(pc) else np.nan,
                mean_matched_crown_iou=np.mean([m[2] for m in accepted]) if accepted else np.nan))
            supports.append(dict(pair_id=pair_id,Plot_name=str(plot.Plot_name),
                circle_id=int(circle.circle_id),geometry=common))
            if disagreement.area>AREA_EPS:
                diffs.append(dict(pair_id=pair_id,Plot_name=str(plot.Plot_name),
                    circle_id=int(circle.circle_id),
                    disagreement_m2=float(disagreement.area),
                    geometry=disagreement))
            for ii,jj,score in accepted:
                crown_matches.append(dict(pair_id=pair_id,plot_crown_local_id=ii,
                    circle_crown_local_id=jj,crown_iou=score))
        except Exception as exc:
            bad.append(dict(Plot_name=str(plot.Plot_name),
                            circle_id=int(circle.circle_id),error=repr(exc)))
if bad:
    pd.DataFrame(bad).to_csv(OVERLAP_DIR/'18A_overlay_errors.csv',index=False)
    raise RuntimeError(f'{len(bad)} plot–circle intersection(s) failed; see 18A_overlay_errors.csv')
OVERLAP=pd.DataFrame(pairs)
assert len(OVERLAP)>0, 'No actual plot–circle intersections found; inspect CRS, source ROI or overlap expectations'
OVERLAP.to_csv(OVERLAP_DIR/'18A_plot_circle_pair_metrics.csv',index=False)
pd.DataFrame(crown_matches,columns=['pair_id','plot_crown_local_id',
    'circle_crown_local_id','crown_iou']).to_csv(OVERLAP_DIR/'18A_crown_matches.csv',index=False)
gpkg=OVERLAP_DIR/'18A_plot_circle_disagreements.gpkg'
if gpkg.exists():gpkg.unlink()
gpd.GeoDataFrame(supports,geometry='geometry',crs=P65.crs).to_file(
    gpkg,layer='shared_plot_circle_support',driver='GPKG')
if diffs:
    gpd.GeoDataFrame(diffs,geometry='geometry',crs=P65.crs).to_file(
        gpkg,layer='musanga_mask_disagreement',driver='GPKG',engine='pyogrio',append=True)
print('Intersecting plot–circle pairs:',len(OVERLAP),
      '| unique plots:',OVERLAP.Plot_name.nunique(),
      '| unique circles:',OVERLAP.circle_id.nunique())
display(OVERLAP.groupby('site').agg(n_pairs=('pair_id','size'),
    mean_abs_cover_difference_pp=('cover_difference_pp',lambda x:np.mean(np.abs(x))),
    mean_mask_iou=('mask_iou','mean'),mean_mask_f1=('mask_f1','mean'),
    total_disagreement_m2=('disagreement_area_m2','sum')).round(3))
display(OVERLAP.sort_values('mask_iou')[[
    'Plot_name','circle_id','shared_area_m2','plot_cover_pct',
    'circle_cover_pct','cover_difference_pp','mask_iou','n_matched_crowns']].head(20).round(3))

# Construct connected components on ALL observed plot-circle overlap edges.
# Each connected component is indivisible under model CV.
nodes=['P:'+str(v) for v in P65.source_id]+['C:'+str(v) for v in C34.circle_id]
parent={x:x for x in nodes}
def root(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]];x=parent[x]
    return x
def join(x,y):
    a,b=root(x),root(y)
    if a!=b:parent[b]=a
for r in OVERLAP.itertuples(index=False):
    join('P:'+str(r.Plot_name),'C:'+str(r.circle_id))
P65['overlap_component']=P65.source_id.map(lambda v:root('P:'+str(v)))
C34['overlap_component']=C34.circle_id.map(lambda v:root('C:'+str(v)))
component_sites={}
for site,g in P65.groupby('site_key'):
    for comp in g.overlap_component:component_sites.setdefault(comp,set()).add(site)
for r in C34.itertuples(index=False):
    component_sites.setdefault(r.overlap_component,set()).add(r.site_key)
mixed={k:v for k,v in component_sites.items() if len(v)>1}
assert not mixed, f'Overlaps or circle site assignments bridge forest groups: {mixed}'
# Spatially order components within each site; assign three folds by geographic position.
components=[]
for site in sorted(set(P65.site_key)|set(C34.site_key)):
    pg=P65[P65.site_key==site]
    cg=C34[C34.site_key==site]
    keys=sorted(set(pg.overlap_component)|set(cg.overlap_component))
    for key in keys:
        polys=list(pg.loc[pg.overlap_component==key].geometry)+list(
            cg.loc[cg.overlap_component==key].geometry)
        center=unary_union(polys).centroid
        components.append(dict(component=key,site=site,x=center.x,y=center.y,
            plots=int((pg.overlap_component==key).sum()),
            circles=int((cg.overlap_component==key).sum())))
COMPONENTS=pd.DataFrame(components)
COMPONENTS['cv_fold']=-1
for site,g in COMPONENTS.groupby('site'):
    xy=g[['x','y']].to_numpy(float)
    projection=PCA(n_components=1).fit_transform(xy).ravel() if len(xy)>1 else np.zeros(len(xy))
    order=np.argsort(projection)
    for fold,idx in enumerate(np.array_split(order,FOLDS65)):
        COMPONENTS.loc[g.index[idx],'cv_fold']=fold
assert COMPONENTS.cv_fold.ge(0).all()
component_fold=COMPONENTS.set_index('component').cv_fold.to_dict()
P65['cv_fold']=P65.overlap_component.map(component_fold).astype(int)
C34['cv_fold']=C34.overlap_component.map(component_fold).astype(int)
assert all(P65.set_index('source_id').loc[r.Plot_name,'cv_fold']==
           C34.set_index('circle_id').loc[r.circle_id,'cv_fold']
           for r in OVERLAP.itertuples(index=False)), 'Overlapping plot/circle leaked across folds'
# For near-but-nonoverlapping polygons, keep 15-m exclusion guard and
# disable any circle whose close plot belongs to a different fold.
C34['cross_fold_plot_intersection']=False
for i,r in C34.iterrows():
    close=P65[(P65.site_key==r.site_key)&
        P65.geometry.intersects(r.geometry.buffer(BUFFER_M))]
    C34.at[i,'cross_fold_plot_intersection']=bool(
        len(close)>0 and any(close.cv_fold!=r.cv_fold))
P65[['source_id','site_key','cv_fold','overlap_component',
    'pct_BA_musanga','area_mismatch','validation_geom_source']].to_csv(
    V4_REF_DIR/'01_field65_plot_folds.csv',index=False)
C34[['circle_id','site_key','cv_fold','overlap_component',
    'nearest_plot','site_inferred','cross_fold_plot_intersection']].to_csv(
    V4_REF_DIR/'01_field65_circle_folds.csv',index=False)
COMPONENTS.to_csv(OVERLAP_DIR/'18A_overlap_components_and_folds.csv',index=False)
print('Connected overlap groups:',len(COMPONENTS),
      '| circle cross-fold proximity flags:',int(C34.cross_fold_plot_intersection.sum()))
display(COMPONENTS.groupby(['site','cv_fold']).agg(
    groups=('component','size'),plots=('plots','sum'),circles=('circles','sum')))
print('ArcGIS GeoPackage:',gpkg)


## 19 — Combined binary supervision and precedence-aware labels
**Priority:** exhaustive field plots (Class_ID 1 = positive, other classes = non-Musanga; remaining plot pixels = negative if within verified plot support), then exhaustive KVO circles (Musanga crowns positive, remainder negative), then partial site crowns (**only specifically annotated crown pixels get a loss weight**). Contradictory overlapping labels become **ignored**, not arbitrarily overwritten. Field plot annotations overlapping held-out plot supports or buffered circles must never leak into training. This function intentionally supersedes the old `label_window` for V4; downstream NB02 must call `label_window_field65` explicitly.

In [ ]:
# NB01.19 — Build V4 raster label components; no mosaic pixel reads.
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box
U_CROWNS=NEW_CROWN.geometry.to_list()
POS_PLOT=NEW_CROWN.loc[NEW_CROWN.is_musanga==1].geometry.to_list()
NEG_PLOT=NEW_CROWN.loc[NEW_CROWN.is_musanga!=1].geometry.to_list()
POS_CIRCLE=E34.geometry.to_list()
# Plot and circle supports stored spatially; actual pixel targets computed on demand.
PLOTS_BY_SITE={s:list(g.geometry) for s,g in P65.groupby('site_key')}
CIRCLES_BY_SITE={s:list(g.geometry) for s,g in C34.groupby('site_key')}
def label_window_field65(mosaic,window,fold=None,role='production'):
    """V4 segmentation label/weight/source contract. role train, val, production.
       source: 0 unknown, 1 partial crown, 2 KVO exhaustive circle, 3 field plot.
       val = heldout exhaustive field plots only, not partial data."""
    assert role in ('train','val','production')
    if role!='production':assert fold in range(FOLDS65)
    d=CACHE[mosaic];site=d['site']
    h,w=int(window.height),int(window.width)
    affine=rasterio.windows.transform(window,d['transform'])
    extent=box(*rasterio.windows.bounds(window,d['transform']))
    def burn(geoms):
        it=[(g,1) for g in geoms if g is not None and not g.is_empty and g.intersects(extent)]
        return rasterize(it,out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if it else np.zeros((h,w),bool)
    roi=burn(d['roi'].geometry)
    Psite=P65[P65.site_key==site]
    Csite=C34[(C34.site_key==site)&~C34.cross_fold_plot_intersection]
    if role=='train':
        ps=Psite[Psite.cv_fold!=fold]
        cs=Csite[Csite.cv_fold!=fold]
    elif role=='val':
        ps=Psite[Psite.cv_fold==fold]
        cs=Csite.iloc[:0] # held-out field plots only as primary validation
    else:
        ps=Psite;cs=Csite
    plot_mask=burn(ps.geometry)
    circle_mask=burn(cs.geometry)&~plot_mask
    positive_plot=burn(POS_PLOT)&plot_mask
    negative_plot=burn(NEG_PLOT)&plot_mask
    positive_circle=burn(POS_CIRCLE)&circle_mask
    partial=d['trees']
    pos_partial=burn(partial.loc[partial._label==1].geometry)
    neg_partial=burn(partial.loc[partial._label==0].geometry)
    partial_support=(pos_partial|neg_partial)&~plot_mask&~circle_mask
    contradiction=(positive_plot&negative_plot)|(
        pos_partial&neg_partial&partial_support)
    # On field plots, background is confirmed non-Musanga by exhaustive annotation.
    target=(positive_plot|positive_circle|(pos_partial&partial_support)).astype('uint8')
    source=np.zeros((h,w),'uint8')
    source[partial_support]=1;source[circle_mask]=2;source[plot_mask]=3
    weights=(roi&(source>0)&~contradiction)
    if role=='val':
        weights=roi&plot_mask&~contradiction
    if role=='train':
        holdout_plots=Psite[Psite.cv_fold==fold]
        holdout_circles=C34[(C34.site_key==site)&
            ((C34.cv_fold==fold)|C34.cross_fold_plot_intersection)]
        forbidden=burn([g.buffer(BUFFER_M) for g in
             list(holdout_plots.geometry)+list(holdout_circles.geometry)])
        weights &= ~forbidden
        # A training raster window itself must not reach inside a holdout support.
        if any(extent.intersects(g.buffer(BUFFER_M)) for g in
               list(holdout_plots.geometry)+list(holdout_circles.geometry)):
            weights[:]=False
    return dict(target=target,weight=weights.astype('uint8'),
                source=source,uncertain=contradiction.astype('uint8'),
                roi=roi.astype('uint8'))
print('V4 label generator ready: plot > circle > partial')


## 20 — Full-field patch candidates, fold manifests and input-ablation experiment design
Sample windows within all 65 field plots and valid KVO circles; preserve earlier partial-crown windows where possible, but **reassign folds and exclude windows touching any held-out plots/circles**. Report each modality's geospatial availability against these frozen footprints. A multimodal comparison is only fair on the **common valid-data support** and exactly the same splits and U-Net architecture. SfM LAS is not a true LiDAR DTM/CHM. The code here prepares reference windows and an experimental design; actual training/ablation must be implemented in NB02, not claimed complete in NB01.

In [ ]:
# NB01.20 — Candidate windows and explicit spatial leakage exclusion
from rasterio.windows import from_bounds, Window
WINDOW_ROWS=[]
def add_windows(gdf,kind,idcol,foldcol='cv_fold'):
    for r in gdf.itertuples(index=False):
        geom=r.geometry;ident=getattr(r,idcol)
        site=r.site_key;fold=int(getattr(r,foldcol))
        for name,entry in CACHE.items():
            if entry['site']!=site:continue
            bounds=entry['footprint']
            if not geom.intersects(bounds):continue
            win=from_bounds(*geom.bounds,transform=entry['transform'])
            rr0=max(0,int(np.floor(win.row_off/PATCH))*PATCH)
            cc0=max(0,int(np.floor(win.col_off/PATCH))*PATCH)
            rr1=min(entry['height'],int(np.ceil((win.row_off+win.height)/PATCH))*PATCH)
            cc1=min(entry['width'],int(np.ceil((win.col_off+win.width)/PATCH))*PATCH)
            for rr in range(rr0,rr1,PATCH):
                for cc in range(cc0,cc1,PATCH):
                    width=min(PATCH,entry['width']-cc);height=min(PATCH,entry['height']-rr)
                    if width<1 or height<1:continue
                    window=Window(cc,rr,width,height)
                    footprint=box(*rasterio.windows.bounds(window,entry['transform']))
                    if not footprint.intersects(geom):continue
                    WINDOW_ROWS.append(dict(site=site,mosaic=name,kind=kind,object_id=str(ident),
                        original_fold=fold,row=int(rr),col=int(cc),width=int(width),height=int(height)))
add_windows(P65,'field_plot','source_id')
add_windows(C34[~C34.cross_fold_plot_intersection],'kvo_circle','circle_id')
# Add legacy PARTIAL windows, but re-check cross-fold guards below.
for r in PATCHES.loc[PATCHES.kind=='partial'].itertuples(index=False):
    WINDOW_ROWS.append(dict(site=r.site,mosaic=r.mosaic,kind='partial_crown',
        object_id='partial',original_fold=-1,row=int(r.row),col=int(r.col),
        width=int(r.width),height=int(r.height)))
WINDOWS65=pd.DataFrame(WINDOW_ROWS).drop_duplicates()
print('Unique candidate windows:',len(WINDOWS65))
WINDOWS65.to_csv(V4_REF_DIR/'01_field65_candidate_windows.csv',index=False)
# Freeze roles and check each window footprint against spatially withheld data.
roles=[]
for fold in range(FOLDS65):
    for r in WINDOWS65.itertuples(index=False):
        entry=CACHE[r.mosaic]
        w=Window(r.col,r.row,r.width,r.height)
        area=box(*rasterio.windows.bounds(w,entry['transform']))
        held_plots=P65[(P65.site_key==r.site)&(P65.cv_fold==fold)]
        held_circles=C34[(C34.site_key==r.site)&
            ((C34.cv_fold==fold)|C34.cross_fold_plot_intersection)]
        guard=any(area.intersects(g.buffer(BUFFER_M)) for g in
                 list(held_plots.geometry)+list(held_circles.geometry))
        val=(r.kind=='field_plot' and r.original_fold==fold)
        split='val' if val else 'excluded_spatial_guard' if guard else 'train'
        # Do not train using other folds' field windows within a held-out region.
        if val:split='val'
        roles.append(dict(cv_fold=fold,site=r.site,mosaic=r.mosaic,kind=r.kind,
            object_id=r.object_id,row=r.row,col=r.col,width=r.width,height=r.height,
            split=split))
SPLITS65=pd.DataFrame(roles).drop_duplicates()
assert len(SPLITS65)>0
assert set(SPLITS65.split).issubset({'train','val','excluded_spatial_guard'})
assert len(SPLITS65[SPLITS65.split=='val'])>0
SPLITS65.to_csv(V4_REF_DIR/'01_field65_cv_window_manifest.csv',index=False)
display(SPLITS65.groupby(['cv_fold','site','kind','split']).size().rename('windows').reset_index())
# Training experiment: each arm must share identical folds and test support.
ARMS=[
    ('A_RGB','RGB',True),
    ('B_RGB_DSM','RGB + SfM DSM local surface morphology',False),
    ('C_RGB_MS','RGB + red/green/red-edge/NIR',False),
    ('D_RGB_LAS','RGB + SfM LAS 3D structural metrics',False),
    ('E_RGB_DSM_MS','RGB + DSM + MS',False),
    ('F_ALL','RGB + DSM + MS + SfM LAS',False)
]
EXPERIMENT=pd.DataFrame(ARMS,columns=['arm','input_channels','baseline'])
EXPERIMENT['same_folds']=True
EXPERIMENT['evaluation']='Held-out field plots; optional independent held-out KVO circles; report per-forest metrics'
EXPERIMENT['support_rule']='Compare on same valid pixels/plots for every paired input experiment'
EXPERIMENT['status']='specified_not_trained'
EXPERIMENT.to_csv(V4_REF_DIR/'01_field65_ablation_design.csv',index=False)
display(EXPERIMENT)
manifest={
    'schema':'musanga_v4_field65_combined_reference_1',
    'input_manifest':str(FIELD65_MANIFEST),
    'field_plot_count':int(len(P65)),'plot_crown_count_retained':int(len(NEW_CROWN)),
    'circle_count':int(len(C34)),'kvo_crown_count_retained':int(len(E34)),
    'fold_count':FOLDS65,'spatial_holdout_buffer_m':BUFFER_M,
    'folds_csv':str(V4_REF_DIR/'01_field65_plot_folds.csv'),
    'cv_manifest':str(V4_REF_DIR/'01_field65_cv_window_manifest.csv'),
    'experiment_design':str(V4_REF_DIR/'01_field65_ablation_design.csv'),
    'annotation_consistency_pairs':str(OVERLAP_DIR/'18A_plot_circle_pair_metrics.csv'),
    'annotation_consistency_gis':str(OVERLAP_DIR/'18A_plot_circle_disagreements.gpkg'),
    'overlap_component_folds':str(OVERLAP_DIR/'18A_overlap_components_and_folds.csv'),
    'plot_circle_overlap_policy':'same connected component same fold; plot labels override circles; circle labels outside plot retained',
    'repeatability_reference':'corrected KVO circle reference, not pure original Nadine first-pass',
    'label_generator':'label_window_field65',
    'legacy_nb01_contract_is_not_authoritative':True,
    'nb02_training_status':'pending',
    'critical_rule':'Never train on any window crossing held-out plot/circle buffered support; maintain plot and circle source precedence'
}
(V4_REF_DIR/'01_field65_reference_contract.json').write_text(json.dumps(manifest,indent=2))
print('Wrote combined V4 field65 reference:',V4_REF_DIR)
